# LightGBM hyperparameter tuning

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

# Determine number of CPU cores for parallel processing (leave one core free)
n_cores = max(1, os.cpu_count() - 1)

# Load data
TRAIN_DATA_PATH = Path("data/train_data.parquet")
train_df = pd.read_parquet(TRAIN_DATA_PATH)

train_df.index = pd.to_numeric(train_df.index)

# Drop datetime features
cols_to_drop = ["issue_d", "earliest_cr_line"]
train_df = train_df.drop(columns=cols_to_drop)

In [ ]:
# Separate features and targets
train_full_X = train_df.drop(["target", "target_annual_roi"], axis=1)
train_full_y_cat = train_df["target"]
train_full_y_reg = train_df["target_annual_roi"]

In [ ]:
# Create subsets of the data for different training sizes
# chronological order is preserved, so we take the last N rows for each subset
train_1k_X = train_full_X.tail(1000)
train_1k_y_cat = train_full_y_cat.tail(1000)
train_1k_y_reg = train_full_y_reg.tail(1000)

train_10k_X = train_full_X.tail(10000)
train_10k_y_cat = train_full_y_cat.tail(10000)
train_10k_y_reg = train_full_y_reg.tail(10000)

train_100k_X = train_full_X.tail(100000)
train_100k_y_cat = train_full_y_cat.tail(100000)
train_100k_y_reg = train_full_y_reg.tail(100000)

## Classification

[Parameters](https://lightgbm.readthedocs.io/en/v4.6.0/pythonapi/lightgbm.LGBMClassifier.html)

In [ ]:
# Add all imports needed for classification hyperparameter tuning
import optuna
import warnings
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

# Check the dates of 1k subset to ensure all data is from same month
# due to reverse order of validation split due to poor results in normal order
print(train_1k_X["issue_d_month"].unique())
print(train_1k_X["issue_d_year"].unique())

# Define path for saving tuning visualizations
LGBM_TUNING_DIR_CAT = Path("hyperparameter_tuning/LightGBM/Classification")

<IntegerArray>
[10]
Length: 1, dtype: Int64
<IntegerArray>
[2016]
Length: 1, dtype: Int64


### 1k

In [ ]:
# Parameter tuning settings
timeout_seconds = 60 * 60 * 0.5
no_improvement_trials = 100

# Split the 1k dataset to evaluate on holdout after tuning
# Had problem with overfitting and extreme bad results on last 200 rows so switched the order
X_tuning = train_1k_X.tail(800)
y_tuning = train_1k_y_cat.tail(800)
X_val_holdout = train_1k_X.head(200)
y_val_holdout = train_1k_y_cat.head(200)


# Define the objective function for Optuna to optimize
def objective_lgbm(trial):
    # Possible parameters to tune with ranges
    params = {
        "boosting_type": trial.suggest_categorical(
            "boosting_type", ["gbdt", "goss"]
        ),  # Type of boosting algorithm - not using "dart" to save time and "dart" did not deliver significantly better results
        "num_leaves": trial.suggest_int(
            "num_leaves", 3, 31
        ),  # Max leaves per tree. Low to prevent memorizing small data
        "max_depth": trial.suggest_int(
            "max_depth", 2, 8
        ),  # Max tree depth. Low to reduce overfitting
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.2, log=True
        ),  # Step size
        "objective": "binary",  # Binary classification objective
        "metric": "auc",  # Evaluation metric
        "scale_pos_weight": trial.suggest_float(
            "scale_pos_weight", 2.0, 10.0
        ),  # Class balancing weight for imbalanced target
        "min_split_gain": trial.suggest_float(
            "min_split_gain", 0, 10.0
        ),  # Min loss reduction required to make a split
        "min_child_weight": trial.suggest_float(
            "min_child_weight", 1e-3, 5.0, log=True
        ),  # Min sum of instance weight needed in a child node
        "min_child_samples": trial.suggest_int(
            "min_child_samples", 15, 100
        ),  # Min data in one leaf
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.3, 0.8
        ),  # % of features used per tree
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-3, 10.0, log=True
        ),  # L1 regularization
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-3, 10.0, log=True
        ),  # L2 regularization
        "random_state": 42,  # Fixed seed for reproducibility
        "n_jobs": 1,  # Use single thread to avoid issues with parallelism in Optuna
        "verbose": -1,  # Suppress LightGBM output
        "colsample_bynode": trial.suggest_float(
            "colsample_bynode", 0.3, 0.8
        ),  # % of features used per node
        "min_data_per_group": trial.suggest_int(
            "min_data_per_group", 10, 100
        ),  # Min data per categorical group
        "max_cat_threshold": trial.suggest_int(
            "max_cat_threshold", 1, 32
        ),  # Max splits for categories
        "cat_l2": trial.suggest_float(
            "cat_l2", 1e-3, 50.0, log=True
        ),  # L2 regularization for categorical splits
        "cat_smooth": trial.suggest_float(
            "cat_smooth", 1e-3, 50.0, log=True
        ),  # Smoothing for categorical splits
        "max_cat_to_onehot": trial.suggest_int(
            "max_cat_to_onehot", 1, 20
        ),  # Threshold for one-hot encoding
        "max_bin": trial.suggest_int("max_bin", 63, 255),  # Max bins for features
        "n_estimators": trial.suggest_int(
            "n_estimators", 100, 1000
        ),  # Number of boosting rounds
    }

    if params["boosting_type"] == "goss":  # Gradient-based One-Side Sampling (GOSS)
        params["subsample"] = 1.0  # LightGBM requirement for GOSS
        params["top_rate"] = trial.suggest_float(
            "top_rate", 0.05, 0.5
        )  # Fraction of large gradients to retain
        params["other_rate"] = trial.suggest_float(
            "other_rate", 0.05, 1.0 - params["top_rate"]
        )  # Fraction of small gradients to retain
    else:  # GBDT (Gradient Boosting Decision Trees)
        params["subsample"] = trial.suggest_float(
            "subsample", 0.5, 1.0
        )  # Row subsampling
        params["subsample_freq"] = trial.suggest_int(
            "subsample_freq", 1, 5
        )  # Frequency of subsampling

    # TimeSeriesSplit to respect temporal order of data 3 splits for 3-fold CV
    tscv = TimeSeriesSplit(n_splits=3)
    cv_scores = []

    # 3-fold CV cycle
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]

        model = LGBMClassifier(**params)

        # Suppress warnings during fitting
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(X_tr, y_tr)

        # Calculate AUC
        preds = model.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))

    # Return mean AUC across CV folds as the objective value to maximize
    return np.mean(cv_scores)


# Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
def stop_optuna(study, trial):
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(
            f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization."
        )
        study.stop()

# Create Optuna study and optimize
study_lgbm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study_lgbm.optimize(
    objective_lgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna]
)

# Print best results and optimal parameters
print("\n" + "=" * 40)
print(f"BEST AUC: {study_lgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_lgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

# Calculate and print parameter importance
print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_lgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("=" * 40)

# Generate and show visualizations of optimization history,
# parameter importance and parallel coordinate plot
fig1 = plot_optimization_history(study_lgbm)
fig1.show()

fig2 = plot_param_importances(study_lgbm)
fig2.show()

fig3 = plot_parallel_coordinate(
    study_lgbm,
    params=[
        "boosting_type",
        "num_leaves",
        "max_depth",
        "learning_rate",
        "scale_pos_weight",
        "min_split_gain",
        "min_child_weight",
        "min_child_samples",
        "colsample_bytree",
        "reg_alpha",
        "reg_lambda",
        "colsample_bynode",
        "min_data_per_group",
        "max_cat_threshold",
        "cat_l2",
        "cat_smooth",
        "max_cat_to_onehot",
        "max_bin",
        "n_estimators",
    ],
)
fig3.show()

# Save visualizations as HTML files
fig1.write_html(LGBM_TUNING_DIR_CAT / "optuna_lgbm_1k_history.html")
fig2.write_html(LGBM_TUNING_DIR_CAT / "optuna_lgbm_1k_importance.html")
fig3.write_html(LGBM_TUNING_DIR_CAT / "optuna_lgbm_1k_parallel.html")

[I 2026-04-25 15:49:54,190] A new study created in memory with name: no-name-ced8b545-8586-4c00-be5a-36565998c76d
[I 2026-04-25 15:49:54,474] Trial 7 finished with value: 0.5712179487179488 and parameters: {'boosting_type': 'goss', 'num_leaves': 21, 'max_depth': 6, 'learning_rate': 0.005375247656768241, 'scale_pos_weight': 4.516906656567825, 'min_split_gain': 0.1103979123847465, 'min_child_weight': 0.1288644627152889, 'min_child_samples': 99, 'colsample_bytree': 0.3670430924546734, 'reg_alpha': 5.530642048994121, 'reg_lambda': 0.01636303917048146, 'colsample_bynode': 0.7896179605623035, 'min_data_per_group': 88, 'max_cat_threshold': 17, 'cat_l2': 0.01808965212360406, 'cat_smooth': 0.03911966843991824, 'max_cat_to_onehot': 17, 'max_bin': 182, 'n_estimators': 181, 'top_rate': 0.13246871236375835, 'other_rate': 0.15335677928729355}. Best is trial 7 with value: 0.5712179487179488.
[I 2026-04-25 15:49:54,490] Trial 0 finished with value: 0.6432829431105292 and parameters: {'boosting_type': 


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 15:50:15,773] Trial 258 finished with value: 0.6625008612077578 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 16, 'max_depth': 5, 'learning_rate': 0.02016817377789622, 'scale_pos_weight': 4.544572321715227, 'min_split_gain': 6.917616788450517, 'min_child_weight': 0.423095009900473, 'min_child_samples': 53, 'colsample_bytree': 0.5263266196552063, 'reg_alpha': 0.3448684222562268, 'reg_lambda': 1.0539130847833778, 'colsample_bynode': 0.4160948607697176, 'min_data_per_group': 91, 'max_cat_threshold': 9, 'cat_l2': 19.573588723961972, 'cat_smooth': 0.007903259024223624, 'max_cat_to_onehot': 3, 'max_bin': 122, 'n_estimators': 553, 'subsample': 0.6286852991497383, 'subsample_freq': 3}. Best is trial 151 with value: 0.6773341409548306.
[I 2026-04-25 15:50:15,790] Trial 256 finished with value: 0.6634348218830978 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 16, 'max_depth': 5, 'learning_rate': 0.018923075141209673, 'scale_pos_weight': 5.287978746234686, 'min_sp


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST AUC: 0.6773
BEST PARAMETERS:
best_params = {
    "boosting_type": "gbdt",
    "num_leaves": 19,
    "max_depth": 6,
    "learning_rate": 0.013118621303382356,
    "scale_pos_weight": 4.234357177755502,
    "min_split_gain": 4.150101489297818,
    "min_child_weight": 0.1439268407905771,
    "min_child_samples": 47,
    "colsample_bytree": 0.6376678823246574,
    "reg_alpha": 0.29085902006459885,
    "reg_lambda": 0.3198616601646805,
    "colsample_bynode": 0.4286302115954977,
    "min_data_per_group": 97,
    "max_cat_threshold": 8,
    "cat_l2": 30.200602897458936,
    "cat_smooth": 0.014021044837121193,
    "max_cat_to_onehot": 6,
    "max_bin": 166,
    "n_estimators": 372,
    "subsample": 0.6440572461920292,
    "subsample_freq": 4,
}

--- PARAMETER IMPORTANCE ---
  min_child_samples   : 0.6012
  min_data_per_group  : 0.1105
  reg_alpha        

In [ ]:
# Check overfit on holdout set with best parameters from tuning
best_params = study_lgbm.best_params.copy()

# Add fixed parameters to best_params and print them
best_params["random_state"] = 42
best_params["n_jobs"] = n_cores
best_params["verbose"] = -1
best_params["objective"] = "binary"
best_params["metric"] = "auc"
print(f"BEST PARAMS: {best_params}")

# Fit on the entire tuning set
final_model = LGBMClassifier(**best_params)

final_model.fit(X_tuning, y_tuning)

# Evaluate on holdout set
holdout_preds = final_model.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

# Print comparison of Optuna CV AUC and holdout test AUC
print(f"Optuna Val AUC: {study_lgbm.best_value:.4f}")
print(f"Holdout Test AUC: {holdout_auc:.4f}")

BEST PARAMS: {'boosting_type': 'gbdt', 'num_leaves': 19, 'max_depth': 6, 'learning_rate': 0.013118621303382356, 'scale_pos_weight': 4.234357177755502, 'min_split_gain': 4.150101489297818, 'min_child_weight': 0.1439268407905771, 'min_child_samples': 47, 'colsample_bytree': 0.6376678823246574, 'reg_alpha': 0.29085902006459885, 'reg_lambda': 0.3198616601646805, 'colsample_bynode': 0.4286302115954977, 'min_data_per_group': 97, 'max_cat_threshold': 8, 'cat_l2': 30.200602897458936, 'cat_smooth': 0.014021044837121193, 'max_cat_to_onehot': 6, 'max_bin': 166, 'n_estimators': 372, 'subsample': 0.6440572461920292, 'subsample_freq': 4, 'random_state': 42, 'n_jobs': -1, 'verbose': -1, 'objective': 'binary', 'metric': 'auc'}
Optuna Val AUC: 0.6773
Holdout Test AUC: 0.6715


### 10k

In [ ]:
# Parameter tuning settings
timeout_seconds = 60 * 60 * 1
no_improvement_trials = 100

# Split the 10k dataset to evaluate on holdout after tuning
X_tuning = train_10k_X.head(8000)
y_tuning = train_10k_y_cat.head(8000)
X_val_holdout = train_10k_X.tail(2000)
y_val_holdout = train_10k_y_cat.tail(2000)


# Define the objective function for Optuna to optimize
def objective_lgbm(trial):
    # Possible parameters to tune with ranges
    params = {
        "boosting_type": trial.suggest_categorical(
            "boosting_type", ["gbdt", "goss"]
        ),  # Type of boosting algorithm - not using "dart" to save time and "dart" did not deliver significantly better results
        "num_leaves": trial.suggest_int(
            "num_leaves", 3, 31
        ),  # Max leaves per tree. Low to prevent memorizing small data
        "max_depth": trial.suggest_int(
            "max_depth", 2, 8
        ),  # Max tree depth. Low to reduce overfitting
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.2, log=True
        ),  # Step size
        "objective": "binary",  # Binary classification objective
        "metric": "auc",  # Evaluation metric
        "scale_pos_weight": trial.suggest_float(
            "scale_pos_weight", 2.0, 10.0
        ),  # Class balancing weight for imbalanced target
        "min_split_gain": trial.suggest_float(
            "min_split_gain", 0, 10.0
        ),  # Min loss reduction required to make a split
        "min_child_weight": trial.suggest_float(
            "min_child_weight", 1e-3, 5.0, log=True
        ),  # Min sum of instance weight needed in a child node
        "min_child_samples": trial.suggest_int(
            "min_child_samples", 15, 100
        ),  # Min data in one leaf
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.3, 0.8
        ),  # % of features used per tree
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-3, 10.0, log=True
        ),  # L1 regularization
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-3, 10.0, log=True
        ),  # L2 regularization
        "random_state": 42,  # Fixed seed for reproducibility
        "n_jobs": 1,  # Use single thread to avoid issues with parallelism in Optuna
        "verbose": -1,  # Suppress LightGBM output
        "colsample_bynode": trial.suggest_float(
            "colsample_bynode", 0.3, 0.8
        ),  # % of features used per node
        "min_data_per_group": trial.suggest_int(
            "min_data_per_group", 10, 100
        ),  # Min data per categorical group
        "max_cat_threshold": trial.suggest_int(
            "max_cat_threshold", 1, 32
        ),  # Max splits for categories
        "cat_l2": trial.suggest_float(
            "cat_l2", 1e-3, 50.0, log=True
        ),  # L2 regularization for categorical splits
        "cat_smooth": trial.suggest_float(
            "cat_smooth", 1e-3, 50.0, log=True
        ),  # Smoothing for categorical splits
        "max_cat_to_onehot": trial.suggest_int(
            "max_cat_to_onehot", 1, 20
        ),  # Threshold for one-hot encoding
        "max_bin": trial.suggest_int("max_bin", 63, 255),  # Max bins for features
        "n_estimators": trial.suggest_int(
            "n_estimators", 100, 1000
        ),  # Number of boosting rounds
    }

    if params["boosting_type"] == "goss":  # Gradient-based One-Side Sampling (GOSS)
        params["subsample"] = 1.0  # LightGBM requirement for GOSS
        params["top_rate"] = trial.suggest_float(
            "top_rate", 0.05, 0.5
        )  # Fraction of large gradients to retain
        params["other_rate"] = trial.suggest_float(
            "other_rate", 0.05, 1.0 - params["top_rate"]
        )  # Fraction of small gradients to retain
    else:  # GBDT (Gradient Boosting Decision Trees)
        params["subsample"] = trial.suggest_float(
            "subsample", 0.5, 1.0
        )  # Row subsampling
        params["subsample_freq"] = trial.suggest_int(
            "subsample_freq", 1, 5
        )  # Frequency of subsampling

    # TimeSeriesSplit to respect temporal order of data 5 splits for 5-fold CV
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []

    # 5-fold CV cycle
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]

        model = LGBMClassifier(**params)

        # Suppress warnings during fitting
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(X_tr, y_tr)

        # Calculate AUC
        preds = model.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))

    # Return mean AUC across CV folds as the objective value to maximize
    return np.mean(cv_scores)


# Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
def stop_optuna(study, trial):
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(
            f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization."
        )
        study.stop()

# Create Optuna study and optimize
study_lgbm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study_lgbm.optimize(
    objective_lgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna]
)

# Print best results and optimal parameters
print("\n" + "=" * 40)
print(f"BEST AUC: {study_lgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_lgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

# Calculate and print parameter importance
print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_lgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("=" * 40)

# Generate and show visualizations of optimization history,
# parameter importance and parallel coordinate plot
fig1 = plot_optimization_history(study_lgbm)
fig1.show()

fig2 = plot_param_importances(study_lgbm)
fig2.show()

fig3 = plot_parallel_coordinate(
    study_lgbm,
    params=[
        "boosting_type",
        "num_leaves",
        "max_depth",
        "learning_rate",
        "scale_pos_weight",
        "min_split_gain",
        "min_child_weight",
        "min_child_samples",
        "colsample_bytree",
        "reg_alpha",
        "reg_lambda",
        "colsample_bynode",
        "min_data_per_group",
        "max_cat_threshold",
        "cat_l2",
        "cat_smooth",
        "max_cat_to_onehot",
        "max_bin",
        "n_estimators",
    ],
)
fig3.show()

# Save visualizations as HTML files
fig1.write_html(LGBM_TUNING_DIR_CAT / "optuna_lgbm_10k_history.html")
fig2.write_html(LGBM_TUNING_DIR_CAT / "optuna_lgbm_10k_importance.html")
fig3.write_html(LGBM_TUNING_DIR_CAT / "optuna_lgbm_10k_parallel.html")

[I 2026-04-25 15:50:20,123] A new study created in memory with name: no-name-0e1b946f-0fcd-465b-90dd-4085697b06e7
[I 2026-04-25 15:50:21,299] Trial 3 finished with value: 0.6913162253492493 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 11, 'max_depth': 2, 'learning_rate': 0.011034963897645229, 'scale_pos_weight': 4.7512119761172835, 'min_split_gain': 1.2168498367172176, 'min_child_weight': 0.049842007916317196, 'min_child_samples': 46, 'colsample_bytree': 0.38934040169873113, 'reg_alpha': 0.028008223591814315, 'reg_lambda': 0.004546462719546609, 'colsample_bynode': 0.36914840908335655, 'min_data_per_group': 86, 'max_cat_threshold': 28, 'cat_l2': 0.11929008623773828, 'cat_smooth': 2.7045166632838926, 'max_cat_to_onehot': 17, 'max_bin': 209, 'n_estimators': 256, 'subsample': 0.7061780324838509, 'subsample_freq': 5}. Best is trial 3 with value: 0.6913162253492493.
[I 2026-04-25 15:50:22,208] Trial 0 finished with value: 0.6928118688488882 and parameters: {'boosting_type': 'goss'


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 15:52:44,759] Trial 346 finished with value: 0.682481314060926 and parameters: {'boosting_type': 'goss', 'num_leaves': 25, 'max_depth': 8, 'learning_rate': 0.031089305715482334, 'scale_pos_weight': 2.8291571000292732, 'min_split_gain': 8.37642098786528, 'min_child_weight': 0.6224545031234456, 'min_child_samples': 66, 'colsample_bytree': 0.35801281225153686, 'reg_alpha': 0.005527471257538076, 'reg_lambda': 1.8316311920817794, 'colsample_bynode': 0.3545273470066443, 'min_data_per_group': 19, 'max_cat_threshold': 2, 'cat_l2': 0.1253427257535439, 'cat_smooth': 0.20252741972869, 'max_cat_to_onehot': 1, 'max_bin': 63, 'n_estimators': 506, 'top_rate': 0.08684658724254779, 'other_rate': 0.11885620906731485}. Best is trial 245 with value: 0.7013105762499531.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 15:52:45,107] Trial 347 finished with value: 0.697677783203694 and parameters: {'boosting_type': 'goss', 'num_leaves': 24, 'max_depth': 8, 'learning_rate': 0.008357192583382083, 'scale_pos_weight': 6.4781357387833625, 'min_split_gain': 7.72269625891477, 'min_child_weight': 0.2563974317964869, 'min_child_samples': 65, 'colsample_bytree': 0.3606242453223351, 'reg_alpha': 0.005365247992282698, 'reg_lambda': 1.24109993466758, 'colsample_bynode': 0.35723141410217407, 'min_data_per_group': 19, 'max_cat_threshold': 1, 'cat_l2': 0.23013810472812773, 'cat_smooth': 0.5217710399079403, 'max_cat_to_onehot': 1, 'max_bin': 63, 'n_estimators': 536, 'top_rate': 0.06297931569529822, 'other_rate': 0.11437443571151815}. Best is trial 245 with value: 0.7013105762499531.
[I 2026-04-25 15:52:45,144] Trial 344 finished with value: 0.6981534146556354 and parameters: {'boosting_type': 'goss', 'num_leaves': 24, 'max_depth': 8, 'learning_rate': 0.009585359741586238, 'scale_pos_weight': 6.1355443499


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 15:52:45,488] Trial 348 finished with value: 0.6951251266227872 and parameters: {'boosting_type': 'goss', 'num_leaves': 25, 'max_depth': 8, 'learning_rate': 0.008344798047580902, 'scale_pos_weight': 6.378948698590474, 'min_split_gain': 7.798742266163628, 'min_child_weight': 0.24289641442745302, 'min_child_samples': 65, 'colsample_bytree': 0.3563903686308207, 'reg_alpha': 0.005256599222385073, 'reg_lambda': 1.3080771053619846, 'colsample_bynode': 0.3522707366869879, 'min_data_per_group': 19, 'max_cat_threshold': 1, 'cat_l2': 1.6207527773687507, 'cat_smooth': 0.3614211084854265, 'max_cat_to_onehot': 1, 'max_bin': 75, 'n_estimators': 525, 'top_rate': 0.060240795540791356, 'other_rate': 0.6884066274386166}. Best is trial 245 with value: 0.7013105762499531.
[I 2026-04-25 15:52:45,652] Trial 349 finished with value: 0.6979947830211344 and parameters: {'boosting_type': 'goss', 'num_leaves': 25, 'max_depth': 8, 'learning_rate': 0.00843648600337008, 'scale_pos_weight': 6.386785190


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 15:52:45,930] Trial 350 finished with value: 0.6953845159267183 and parameters: {'boosting_type': 'goss', 'num_leaves': 25, 'max_depth': 8, 'learning_rate': 0.008313886763585104, 'scale_pos_weight': 6.8374203652040695, 'min_split_gain': 8.776620444737008, 'min_child_weight': 0.6435010292472431, 'min_child_samples': 65, 'colsample_bytree': 0.3586630388729784, 'reg_alpha': 0.00561221722672104, 'reg_lambda': 1.2279248109170238, 'colsample_bynode': 0.34017770617817794, 'min_data_per_group': 16, 'max_cat_threshold': 1, 'cat_l2': 0.20908455050213204, 'cat_smooth': 0.36723714051701, 'max_cat_to_onehot': 1, 'max_bin': 76, 'n_estimators': 527, 'top_rate': 0.050920908834292025, 'other_rate': 0.6136774896006213}. Best is trial 245 with value: 0.7013105762499531.



[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST AUC: 0.7013
BEST PARAMETERS:
best_params = {
    "boosting_type": "goss",
    "num_leaves": 26,
    "max_depth": 8,
    "learning_rate": 0.011490524171318373,
    "scale_pos_weight": 2.5548028672067264,
    "min_split_gain": 8.402184412779064,
    "min_child_weight": 0.7889913973072552,
    "min_child_samples": 67,
    "colsample_bytree": 0.3322094184323836,
    "reg_alpha": 0.009195021998688829,
    "reg_lambda": 1.241109986317591,
    "colsample_bynode": 0.3432164431126759,
    "min_data_per_group": 13,
    "max_cat_threshold": 3,
    "cat_l2": 4.82194592281236,
    "cat_smooth": 0.41553169506659904,
    "max_cat_to_onehot": 1,
    "max_bin": 66,
    "n_estimators": 511,
    "top_rate": 0.07974625682135272,
    "other_rate": 0.15560637534232696,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.5685
  num_leaves          : 0.0788
  colsample_bytree    : 0.0543
  max_cat_to_onehot   : 0.0503
  min_split_g

In [ ]:
# Check overfit on holdout set with best parameters from tuning
best_params = study_lgbm.best_params.copy()

# Apply scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]
best_params["n_estimators"] = original_trees * 2
best_params["learning_rate"] = original_lr / 2
print(
    f"\n[Scaling Trick Applied] {best_params['boosting_type'].upper()}: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}"
)

# Add fixed parameters to best_params and print them
best_params["random_state"] = 42
best_params["n_jobs"] = n_cores
best_params["verbose"] = -1
best_params["objective"] = "binary"
best_params["metric"] = "auc"
print(f"BEST PARAMS: {best_params}")

final_model = LGBMClassifier(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning)

# Evaluate on holdout set
holdout_preds = final_model.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

# Print comparison of Optuna CV AUC and holdout test AUC
print("\n" + "=" * 40)
print(f"Optuna Val AUC:   {study_lgbm.best_value:.4f}")
print(f"Holdout Test AUC: {holdout_auc:.4f}")
print("=" * 40)


[Scaling Trick Applied] GOSS: Trees 511 -> 1022, LR 0.0115 -> 0.0057
BEST PARAMS: {'boosting_type': 'goss', 'num_leaves': 26, 'max_depth': 8, 'learning_rate': 0.0057452620856591865, 'scale_pos_weight': 2.5548028672067264, 'min_split_gain': 8.402184412779064, 'min_child_weight': 0.7889913973072552, 'min_child_samples': 67, 'colsample_bytree': 0.3322094184323836, 'reg_alpha': 0.009195021998688829, 'reg_lambda': 1.241109986317591, 'colsample_bynode': 0.3432164431126759, 'min_data_per_group': 13, 'max_cat_threshold': 3, 'cat_l2': 4.82194592281236, 'cat_smooth': 0.41553169506659904, 'max_cat_to_onehot': 1, 'max_bin': 66, 'n_estimators': 1022, 'top_rate': 0.07974625682135272, 'other_rate': 0.15560637534232696, 'random_state': 42, 'n_jobs': -1, 'verbose': -1, 'objective': 'binary', 'metric': 'auc'}

Optuna Val AUC:   0.7013
Holdout Test AUC: 0.6986


### 100k

In [ ]:
# Parameter tuning settings
timeout_seconds = 60 * 60 * 3
no_improvement_trials = 100

# Split the 100k dataset to evaluate on holdout after tuning
X_tuning = train_100k_X.head(80000)
y_tuning = train_100k_y_cat.head(80000)
X_val_holdout = train_100k_X.tail(20000)
y_val_holdout = train_100k_y_cat.tail(20000)


# Define the objective function for Optuna to optimize
def objective_lgbm(trial):
    # Possible parameters to tune with ranges
    params = {
        "boosting_type": trial.suggest_categorical(
            "boosting_type", ["gbdt", "goss"]
        ),  # Type of boosting algorithm - not using "dart" to save time and "dart" did not deliver significantly better results
        "num_leaves": trial.suggest_int("num_leaves", 5, 512),  # Max leaves per tree
        "max_depth": trial.suggest_int("max_depth", -1, 20),  # Max tree depth
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.2, log=True
        ),  # Step size
        "objective": "binary",  # Binary classification objective
        "metric": "auc",  # Evaluation metric
        "scale_pos_weight": trial.suggest_float(
            "scale_pos_weight", 2, 10
        ),  # Class balancing weight for imbalanced target
        "min_split_gain": trial.suggest_float(
            "min_split_gain", 0, 30.0
        ),  # Min loss reduction required to make a split
        "min_child_weight": trial.suggest_float(
            "min_child_weight", 1e-5, 10.0, log=True
        ),  # Min sum of instance weight needed in a child node
        "min_child_samples": trial.suggest_int(
            "min_child_samples", 10, 500
        ),  # Min data in one leaf
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.2, 1.0
        ),  # % of features used per tree
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-8, 100.0, log=True
        ),  # L1 regularization
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-8, 100.0, log=True
        ),  # L2 regularization
        "random_state": 42,  # Fixed seed for reproducibility
        "n_jobs": 1,  # Use single thread to avoid issues with parallelism in Optuna
        "verbose": -1,  # Suppress LightGBM output
        "colsample_bynode": trial.suggest_float(
            "colsample_bynode", 0.2, 1.0
        ),  # % of features used per node
        "min_data_per_group": trial.suggest_int(
            "min_data_per_group", 1, 1000
        ),  # Min data per categorical group
        "max_cat_threshold": trial.suggest_int(
            "max_cat_threshold", 1, 1000
        ),  # Max splits for categories
        "cat_l2": trial.suggest_float(
            "cat_l2", 1e-8, 100.0, log=True
        ),  # L2 regularization for categorical splits
        "cat_smooth": trial.suggest_float(
            "cat_smooth", 1e-8, 100.0, log=True
        ),  # Smoothing for categorical splits
        "max_cat_to_onehot": trial.suggest_int(
            "max_cat_to_onehot", 1, 51
        ),  # Threshold for one-hot encoding
        "max_bin": trial.suggest_int("max_bin", 63, 511),  # Max bins for features
        "n_estimators": trial.suggest_int(
            "n_estimators", 100, 1000
        ),  # Number of boosting rounds
    }

    if params["boosting_type"] == "goss":  # Gradient-based One-Side Sampling (GOSS)
        params["subsample"] = 1.0  # LightGBM requirement for GOSS
        params["top_rate"] = trial.suggest_float(
            "top_rate", 0.05, 0.8
        )  # Fraction of large gradients to retain
        params["other_rate"] = trial.suggest_float(
            "other_rate", 0.05, 1.0 - params["top_rate"]
        )  # Fraction of small gradients to retain
    else:
        params["subsample"] = trial.suggest_float(
            "subsample", 0.1, 1.0
        )  # Row subsampling
        params["subsample_freq"] = trial.suggest_int(
            "subsample_freq", 1, 10
        )  # Frequency of subsampling

    # TimeSeriesSplit to respect temporal order of data 5 splits for 5-fold CV
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []

    # 5-fold CV cycle
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]

        model = LGBMClassifier(**params)

        # Suppress warnings during fitting
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(X_tr, y_tr)

        # Calculate AUC
        preds = model.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))

    # Return mean AUC across CV folds as the objective value to maximize
    return np.mean(cv_scores)


# Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
def stop_optuna(study, trial):
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(
            f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization."
        )
        study.stop()


# Create Optuna study and optimize
study_lgbm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study_lgbm.optimize(
    objective_lgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna]
)

# Print best results and optimal parameters
print("\n" + "=" * 40)
print(f"BEST AUC: {study_lgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_lgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

# Calculate and print parameter importance
print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_lgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("=" * 40)

# Generate and show visualizations of optimization history,
# parameter importance and parallel coordinate plot
fig1 = plot_optimization_history(study_lgbm)
fig1.show()

fig2 = plot_param_importances(study_lgbm)
fig2.show()

fig3 = plot_parallel_coordinate(
    study_lgbm,
    params=[
        "boosting_type",
        "num_leaves",
        "max_depth",
        "learning_rate",
        "scale_pos_weight",
        "min_split_gain",
        "min_child_weight",
        "min_child_samples",
        "colsample_bytree",
        "reg_alpha",
        "reg_lambda",
        "colsample_bynode",
        "min_data_per_group",
        "max_cat_threshold",
        "cat_l2",
        "cat_smooth",
        "max_cat_to_onehot",
        "max_bin",
        "n_estimators",
    ],
)
fig3.show()

# Save visualizations as HTML files
fig1.write_html(LGBM_TUNING_DIR_CAT / "optuna_lgbm_100k_history.html")
fig2.write_html(LGBM_TUNING_DIR_CAT / "optuna_lgbm_100k_importance.html")
fig3.write_html(LGBM_TUNING_DIR_CAT / "optuna_lgbm_100k_parallel.html")

[I 2026-04-25 15:52:54,717] A new study created in memory with name: no-name-f9d52ece-027d-4720-8100-fbe0f73c5d4e
[I 2026-04-25 15:53:15,646] Trial 4 finished with value: 0.7039879129522958 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 41, 'max_depth': 6, 'learning_rate': 0.07382190803057405, 'scale_pos_weight': 3.529092898526666, 'min_split_gain': 23.190103058564844, 'min_child_weight': 0.012876210208074489, 'min_child_samples': 305, 'colsample_bytree': 0.9431537409012714, 'reg_alpha': 5.698812673178861e-07, 'reg_lambda': 4.614241724882315e-05, 'colsample_bynode': 0.25704899552163574, 'min_data_per_group': 620, 'max_cat_threshold': 99, 'cat_l2': 4.50744967453117e-06, 'cat_smooth': 0.20703022656484735, 'max_cat_to_onehot': 27, 'max_bin': 385, 'n_estimators': 362, 'subsample': 0.9579037649496904, 'subsample_freq': 5}. Best is trial 4 with value: 0.7039879129522958.
[I 2026-04-25 15:53:19,907] Trial 1 finished with value: 0.6978547249787836 and parameters: {'boosting_type': 'gb


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 16:43:18,808] Trial 420 finished with value: 0.7090748067860579 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 402, 'max_depth': 19, 'learning_rate': 0.012064509498672367, 'scale_pos_weight': 6.000820536995601, 'min_split_gain': 14.744267543389505, 'min_child_weight': 0.00042452833799367577, 'min_child_samples': 231, 'colsample_bytree': 0.27082014638516005, 'reg_alpha': 0.31773748021833764, 'reg_lambda': 0.28807620257771754, 'colsample_bynode': 0.2544397541629257, 'min_data_per_group': 873, 'max_cat_threshold': 851, 'cat_l2': 0.00010428825682054862, 'cat_smooth': 6.354523678399262e-05, 'max_cat_to_onehot': 42, 'max_bin': 262, 'n_estimators': 917, 'subsample': 0.8070010172623305, 'subsample_freq': 4}. Best is trial 313 with value: 0.7133944066754897.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 16:43:23,527] Trial 414 finished with value: 0.7124159758925691 and parameters: {'boosting_type': 'goss', 'num_leaves': 407, 'max_depth': 19, 'learning_rate': 0.011853753666117186, 'scale_pos_weight': 2.006428308993827, 'min_split_gain': 2.61457185990671, 'min_child_weight': 0.0014665052132248712, 'min_child_samples': 249, 'colsample_bytree': 0.3453705061879384, 'reg_alpha': 0.29051633589300185, 'reg_lambda': 0.1764762783088752, 'colsample_bynode': 0.2517626085576692, 'min_data_per_group': 841, 'max_cat_threshold': 846, 'cat_l2': 0.00011517358561435425, 'cat_smooth': 2.3811404426187146e-05, 'max_cat_to_onehot': 38, 'max_bin': 171, 'n_estimators': 982, 'top_rate': 0.12642323132058134, 'other_rate': 0.432581154769897}. Best is trial 313 with value: 0.7133944066754897.


...


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 16:43:39,232] Trial 415 finished with value: 0.7023183054340401 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 404, 'max_depth': 19, 'learning_rate': 0.04994425792630373, 'scale_pos_weight': 6.129854694140937, 'min_split_gain': 1.38604714656651, 'min_child_weight': 0.00038079338817436617, 'min_child_samples': 247, 'colsample_bytree': 0.25828743719334085, 'reg_alpha': 0.3221174318459293, 'reg_lambda': 0.19585247883772386, 'colsample_bynode': 0.25358630061866566, 'min_data_per_group': 907, 'max_cat_threshold': 848, 'cat_l2': 0.00011947582860669784, 'cat_smooth': 8.069251302818221e-05, 'max_cat_to_onehot': 42, 'max_bin': 205, 'n_estimators': 959, 'subsample': 0.9125659664217687, 'subsample_freq': 2}. Best is trial 313 with value: 0.7133944066754897.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 16:44:05,346] Trial 419 finished with value: 0.7121240496180189 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 403, 'max_depth': 19, 'learning_rate': 0.01187057534666296, 'scale_pos_weight': 5.882966767230629, 'min_split_gain': 1.1314509698807735, 'min_child_weight': 0.00043851962596906565, 'min_child_samples': 248, 'colsample_bytree': 0.2751185538343298, 'reg_alpha': 0.31304603766252603, 'reg_lambda': 0.23570197045556773, 'colsample_bynode': 0.25570255457345836, 'min_data_per_group': 836, 'max_cat_threshold': 320, 'cat_l2': 0.00011404091336086746, 'cat_smooth': 8.632701913644228e-05, 'max_cat_to_onehot': 42, 'max_bin': 201, 'n_estimators': 970, 'subsample': 0.9104826683186029, 'subsample_freq': 4}. Best is trial 313 with value: 0.7133944066754897.



[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST AUC: 0.7134
BEST PARAMETERS:
best_params = {
    "boosting_type": "goss",
    "num_leaves": 444,
    "max_depth": 15,
    "learning_rate": 0.013490712742580084,
    "scale_pos_weight": 2.529589934768151,
    "min_split_gain": 3.4510864669147536,
    "min_child_weight": 0.19239271728859866,
    "min_child_samples": 234,
    "colsample_bytree": 0.31324120666189326,
    "reg_alpha": 0.6545836243558268,
    "reg_lambda": 0.1069421692665621,
    "colsample_bynode": 0.21348019642019067,
    "min_data_per_group": 803,
    "max_cat_threshold": 819,
    "cat_l2": 0.00011204200863439848,
    "cat_smooth": 1.5621591746591863,
    "max_cat_to_onehot": 36,
    "max_bin": 258,
    "n_estimators": 872,
    "top_rate": 0.24249410683009498,
    "other_rate": 0.42369427528780756,
}

--- PARAMETER IMPORTANCE ---
  min_split_gain      : 0.2649
  boosting_type       : 0.1363
  reg_alpha           : 0.1359
  n_estimators        : 0.102

In [ ]:
# Check overfit on holdout set with best parameters from tuning
best_params = study_lgbm.best_params.copy()

# Apply scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]
best_params["n_estimators"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10
print(
    f"\n[Scaling Trick Applied] {best_params['boosting_type'].upper()}: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}"
)


# Add fixed parameters to best_params and print them
best_params["random_state"] = 42
best_params["n_jobs"] = n_cores
best_params["verbose"] = -1
best_params["objective"] = "binary"
best_params["metric"] = "auc"
print(f"BEST PARAMS: {best_params}")

final_model = LGBMClassifier(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning)

# Evaluate on holdout set
holdout_preds = final_model.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

# Print comparison of Optuna CV AUC and holdout test AUC
print("\n" + "=" * 40)
print(f"Optuna Val AUC:   {study_lgbm.best_value:.4f}")
print(f"Holdout Test AUC: {holdout_auc:.4f}")
print("=" * 40)


[Scaling Trick Applied] GOSS: Trees 872 -> 8720, LR 0.0135 -> 0.0013
BEST PARAMS: {'boosting_type': 'goss', 'num_leaves': 444, 'max_depth': 15, 'learning_rate': 0.0013490712742580085, 'scale_pos_weight': 2.529589934768151, 'min_split_gain': 3.4510864669147536, 'min_child_weight': 0.19239271728859866, 'min_child_samples': 234, 'colsample_bytree': 0.31324120666189326, 'reg_alpha': 0.6545836243558268, 'reg_lambda': 0.1069421692665621, 'colsample_bynode': 0.21348019642019067, 'min_data_per_group': 803, 'max_cat_threshold': 819, 'cat_l2': 0.00011204200863439848, 'cat_smooth': 1.5621591746591863, 'max_cat_to_onehot': 36, 'max_bin': 258, 'n_estimators': 8720, 'top_rate': 0.24249410683009498, 'other_rate': 0.42369427528780756, 'random_state': 42, 'n_jobs': -1, 'verbose': -1, 'objective': 'binary', 'metric': 'auc'}

Optuna Val AUC:   0.7134
Holdout Test AUC: 0.7169


### Whole training data set

In [ ]:
# Parameter tuning settings
timeout_seconds = 60 * 60 * 4
no_improvement_trials = 100

# Split the dataset to evaluate on holdout after tuning
split_index = int(len(train_full_X) * 0.8)
X_tuning = train_full_X[:split_index]
y_tuning = train_full_y_cat[:split_index]
X_val_holdout = train_full_X[split_index:]
y_val_holdout = train_full_y_cat[split_index:]


# Define the objective function for Optuna to optimize
def objective_lgbm(trial):
    # Possible parameters to tune with ranges
    params = {
        "boosting_type": trial.suggest_categorical(
            "boosting_type", ["gbdt", "goss"]
        ),  # Type of boosting algorithm - not using "dart" to save time and "dart" did not deliver significantly better results
        "num_leaves": trial.suggest_int("num_leaves", 5, 512),  # Max leaves per tree
        "max_depth": trial.suggest_int("max_depth", -1, 20),  # Max tree depth
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.2, log=True
        ),  # Step size
        "objective": "binary",  # Binary classification objective
        "metric": "auc",  # Evaluation metric
        "scale_pos_weight": trial.suggest_float(
            "scale_pos_weight", 2, 10
        ),  # Class balancing weight for imbalanced target
        "min_split_gain": trial.suggest_float(
            "min_split_gain", 0, 30.0
        ),  # Min loss reduction required to make a split
        "min_child_weight": trial.suggest_float(
            "min_child_weight", 1e-5, 10.0, log=True
        ),  # Min sum of instance weight needed in a child node
        "min_child_samples": trial.suggest_int(
            "min_child_samples", 10, 500
        ),  # Min data in one leaf
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.2, 1.0
        ),  # % of features used per tree
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-8, 100.0, log=True
        ),  # L1 regularization
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-8, 100.0, log=True
        ),  # L2 regularization
        "random_state": 42,  # Fixed seed for reproducibility
        "n_jobs": 1,  # Use single thread to avoid issues with parallelism in Optuna
        "verbose": -1,  # Suppress LightGBM output
        "colsample_bynode": trial.suggest_float(
            "colsample_bynode", 0.2, 1.0
        ),  # % of features used per node
        "min_data_per_group": trial.suggest_int(
            "min_data_per_group", 1, 1000
        ),  # Min data per categorical group
        "max_cat_threshold": trial.suggest_int(
            "max_cat_threshold", 1, 1000
        ),  # Max splits for categories
        "cat_l2": trial.suggest_float(
            "cat_l2", 1e-8, 100.0, log=True
        ),  # L2 regularization for categorical splits
        "cat_smooth": trial.suggest_float(
            "cat_smooth", 1e-8, 100.0, log=True
        ),  # Smoothing for categorical splits
        "max_cat_to_onehot": trial.suggest_int(
            "max_cat_to_onehot", 1, 51
        ),  # Threshold for one-hot encoding
        "max_bin": trial.suggest_int("max_bin", 63, 511),  # Max bins for features
        "n_estimators": trial.suggest_int(
            "n_estimators", 100, 1000
        ),  # Number of boosting rounds
    }

    if params["boosting_type"] == "goss":  # Gradient-based One-Side Sampling (GOSS)
        params["subsample"] = 1.0  # LightGBM requirement for GOSS
        params["top_rate"] = trial.suggest_float(
            "top_rate", 0.05, 0.8
        )  # Fraction of large gradients to retain
        params["other_rate"] = trial.suggest_float(
            "other_rate", 0.05, 1.0 - params["top_rate"]
        )  # Fraction of small gradients to retain
    else:
        params["subsample"] = trial.suggest_float(
            "subsample", 0.1, 1.0
        )  # Row subsampling
        params["subsample_freq"] = trial.suggest_int(
            "subsample_freq", 1, 10
        )  # Frequency of subsampling

    # TimeSeriesSplit to respect temporal order of data 3 splits for 3-fold CV
    tscv = TimeSeriesSplit(n_splits=3)
    cv_scores = []

    # 3-fold CV cycle
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]

        model = LGBMClassifier(**params)

        # Suppress warnings during fitting
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(X_tr, y_tr)

        # Calculate AUC
        preds = model.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))

    # Return mean AUC across CV folds as the objective value to maximize
    return np.mean(cv_scores)


# Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
def stop_optuna(study, trial):
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(
            f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization."
        )
        study.stop()

# Create Optuna study and optimize
study_lgbm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study_lgbm.optimize(
    objective_lgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna]
)

# Print best results and optimal parameters
print("\n" + "=" * 40)
print(f"BEST AUC: {study_lgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_lgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

# Calculate and print parameter importance
print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_lgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("=" * 40)

# Generate and show visualizations of optimization history,
# parameter importance and parallel coordinate plot
fig1 = plot_optimization_history(study_lgbm)
fig1.show()

fig2 = plot_param_importances(study_lgbm)
fig2.show()

fig3 = plot_parallel_coordinate(
    study_lgbm,
    params=[
        "boosting_type",
        "num_leaves",
        "max_depth",
        "learning_rate",
        "scale_pos_weight",
        "min_split_gain",
        "min_child_weight",
        "min_child_samples",
        "colsample_bytree",
        "reg_alpha",
        "reg_lambda",
        "colsample_bynode",
        "min_data_per_group",
        "max_cat_threshold",
        "cat_l2",
        "cat_smooth",
        "max_cat_to_onehot",
        "max_bin",
        "n_estimators",
    ],
)
fig3.show()

# Save visualizations as HTML files
fig1.write_html(LGBM_TUNING_DIR_CAT / "optuna_lgbm_full_history.html")
fig2.write_html(LGBM_TUNING_DIR_CAT / "optuna_lgbm_full_importance.html")
fig3.write_html(LGBM_TUNING_DIR_CAT / "optuna_lgbm_full_parallel.html")

[I 2026-04-25 16:46:50,843] A new study created in memory with name: no-name-b5a904b2-cce6-4933-9fef-b41a3dec45ea


[I 2026-04-25 16:48:26,885] Trial 7 finished with value: 0.7313784756171374 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 470, 'max_depth': 2, 'learning_rate': 0.06864542579365267, 'scale_pos_weight': 7.762779177414871, 'min_split_gain': 2.8621739571144413, 'min_child_weight': 0.0006621577859487494, 'min_child_samples': 205, 'colsample_bytree': 0.41640636205924486, 'reg_alpha': 3.8999094146303785, 'reg_lambda': 2.9664304014079545e-05, 'colsample_bynode': 0.9760590838044547, 'min_data_per_group': 532, 'max_cat_threshold': 383, 'cat_l2': 3.0170725715281244e-07, 'cat_smooth': 1.1646355604702144e-08, 'max_cat_to_onehot': 35, 'max_bin': 66, 'n_estimators': 227, 'subsample': 0.6425236906943967, 'subsample_freq': 2}. Best is trial 7 with value: 0.7313784756171374.
[I 2026-04-25 16:49:57,647] Trial 2 finished with value: 0.7342846827787698 and parameters: {'boosting_type': 'goss', 'num_leaves': 96, 'max_depth': 15, 'learning_rate': 0.11901157219045892, 'scale_pos_weight': 9.791532150


BEST AUC: 0.7389
BEST PARAMETERS:
best_params = {
    "boosting_type": "gbdt",
    "num_leaves": 176,
    "max_depth": 18,
    "learning_rate": 0.013263394883988124,
    "scale_pos_weight": 5.720161074894982,
    "min_split_gain": 7.015394333142449,
    "min_child_weight": 0.7877329829167548,
    "min_child_samples": 287,
    "colsample_bytree": 0.6513068572006349,
    "reg_alpha": 0.25347089784492205,
    "reg_lambda": 5.51570221223381e-05,
    "colsample_bynode": 0.33764943797808045,
    "min_data_per_group": 550,
    "max_cat_threshold": 187,
    "cat_l2": 2.2848847335233635e-07,
    "cat_smooth": 1.2867859076903958e-05,
    "max_cat_to_onehot": 46,
    "max_bin": 127,
    "n_estimators": 967,
    "subsample": 0.7854117062367028,
    "subsample_freq": 10,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.3702
  max_bin             : 0.2223
  min_data_per_group  : 0.0903
  max_depth           : 0.0786
  num_leaves          : 0.0654
  min_split_gain      : 0.0536
  max_cat_th

In [ ]:
# Check overfit on holdout set with best parameters from tuning
best_params = study_lgbm.best_params.copy()

# Apply scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]
best_params["n_estimators"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10
print(
    f"\n[Scaling Trick Applied] {best_params['boosting_type'].upper()}: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}"
)


# Add fixed parameters to best_params and print them
best_params["random_state"] = 42
best_params["n_jobs"] = n_cores
best_params["verbose"] = -1
best_params["objective"] = "binary"
best_params["metric"] = "auc"
print(f"BEST PARAMS: {best_params}")

final_model = LGBMClassifier(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning)

# Evaluate on holdout set
holdout_preds = final_model.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

# Print comparison of Optuna CV AUC and holdout test AUC
print("\n" + "=" * 40)
print(f"Optuna Val AUC:   {study_lgbm.best_value:.4f}")
print(f"Holdout Test AUC: {holdout_auc:.4f}")
print("=" * 40)


[Scaling Trick Applied] GBDT: Trees 967 -> 9670, LR 0.0133 -> 0.0013
BEST PARAMS: {'boosting_type': 'gbdt', 'num_leaves': 176, 'max_depth': 18, 'learning_rate': 0.0013263394883988124, 'scale_pos_weight': 5.720161074894982, 'min_split_gain': 7.015394333142449, 'min_child_weight': 0.7877329829167548, 'min_child_samples': 287, 'colsample_bytree': 0.6513068572006349, 'reg_alpha': 0.25347089784492205, 'reg_lambda': 5.51570221223381e-05, 'colsample_bynode': 0.33764943797808045, 'min_data_per_group': 550, 'max_cat_threshold': 187, 'cat_l2': 2.2848847335233635e-07, 'cat_smooth': 1.2867859076903958e-05, 'max_cat_to_onehot': 46, 'max_bin': 127, 'n_estimators': 9670, 'subsample': 0.7854117062367028, 'subsample_freq': 10, 'random_state': 42, 'n_jobs': -1, 'verbose': -1, 'objective': 'binary', 'metric': 'auc'}

Optuna Val AUC:   0.7389
Holdout Test AUC: 0.7286


## Regression

[Parameters](https://lightgbm.readthedocs.io/en/v4.6.0/pythonapi/lightgbm.LGBMRegressor.html)

In [ ]:
# Import necessary libraries for regression tuning and evaluation
import optuna
import numpy as np
import warnings
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

# Define path for saving tuning visualizations
LGBM_TUNING_DIR_REG = Path("hyperparameter_tuning/LightGBM/Regression")

### 1k

In [ ]:
# Parameter tuning settings
timeout_seconds = 60 * 60 * 0.5
no_improvement_trials = 100

# Split the 1k dataset to evaluate on holdout after tuning
# Had problem with overfitting and extreme bad results on last 200 rows so switched the order
X_tuning = train_1k_X.tail(800)
y_tuning = train_1k_y_reg.tail(800)
X_val_holdout = train_1k_X.head(200)
y_val_holdout = train_1k_y_reg.head(200)


# Define the objective function for Optuna to optimize
def objective_lgbm(trial):
    # Possible parameters to tune with ranges
    params = {
        "boosting_type": trial.suggest_categorical(
            "boosting_type", ["gbdt", "goss"]
        ),  # Type of boosting algorithm - not using "dart" to save time and "dart" did not deliver significantly better results
        "num_leaves": trial.suggest_int(
            "num_leaves", 3, 31
        ),  # Max leaves per tree. Low to prevent memorizing small data
        "max_depth": trial.suggest_int(
            "max_depth", 2, 8
        ),  # Max tree depth. Low to reduce overfitting
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.2, log=True
        ),  # Step size
        "objective": "regression",  # Regression objective
        "metric": "rmse",  # Evaluation metric
        "min_split_gain": trial.suggest_float(
            "min_split_gain", 0, 10.0
        ),  # Min loss reduction required to make a split
        "min_child_weight": trial.suggest_float(
            "min_child_weight", 1e-3, 5.0, log=True
        ),  # Min sum of instance weight needed in a child node
        "min_child_samples": trial.suggest_int(
            "min_child_samples", 15, 100
        ),  # Min data in one leaf
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.3, 0.8
        ),  # % of features used per tree
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-3, 10.0, log=True
        ),  # L1 regularization
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-3, 10.0, log=True
        ),  # L2 regularization
        "random_state": 42,  # Fixed seed for reproducibility
        "n_jobs": 1,  # Use single thread to avoid issues with parallelism in Optuna
        "verbose": -1,  # Suppress LightGBM output
        "colsample_bynode": trial.suggest_float(
            "colsample_bynode", 0.3, 0.8
        ),  # % of features used per node
        "min_data_per_group": trial.suggest_int(
            "min_data_per_group", 10, 100
        ),  # Min data per categorical group
        "max_cat_threshold": trial.suggest_int(
            "max_cat_threshold", 1, 32
        ),  # Max splits for categories
        "cat_l2": trial.suggest_float(
            "cat_l2", 1e-3, 50.0, log=True
        ),  # L2 regularization for categorical splits
        "cat_smooth": trial.suggest_float(
            "cat_smooth", 1e-3, 50.0, log=True
        ),  # Smoothing for categorical splits
        "max_cat_to_onehot": trial.suggest_int(
            "max_cat_to_onehot", 1, 20
        ),  # Threshold for one-hot encoding
        "max_bin": trial.suggest_int("max_bin", 63, 255),  # Max bins for features
        "n_estimators": trial.suggest_int(
            "n_estimators", 100, 1000
        ),  # Number of boosting rounds
    }

    if params["boosting_type"] == "goss":  # Gradient-based One-Side Sampling (GOSS)
        params["subsample"] = 1.0  # LightGBM requirement for GOSS
        params["top_rate"] = trial.suggest_float(
            "top_rate", 0.05, 0.5
        )  # Fraction of large gradients to retain
        params["other_rate"] = trial.suggest_float(
            "other_rate", 0.05, 1.0 - params["top_rate"]
        )  # Fraction of small gradients to retain
    else:  # GBDT (Gradient Boosting Decision Trees)
        params["subsample"] = trial.suggest_float(
            "subsample", 0.5, 1.0
        )  # Row subsampling
        params["subsample_freq"] = trial.suggest_int(
            "subsample_freq", 1, 5
        )  # Frequency of subsampling

    # TimeSeriesSplit to respect temporal order of data 3 splits for 3-fold CV
    tscv = TimeSeriesSplit(n_splits=3)
    cv_scores = []

    # # 3-fold CV cycle
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]

        model = LGBMRegressor(**params)

        # Suppress warnings during fitting
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(X_tr, y_tr)

        # Calculate RMSE
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)

    # Return mean RMSE across CV folds as the objective value to minimize
    return np.mean(cv_scores)


# Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
def stop_optuna(study, trial):
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(
            f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization."
        )
        study.stop()


# Create Optuna study and optimize
study_lgbm = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_lgbm.optimize(
    objective_lgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna]
)

# Print best results and optimal parameters
print("\n" + "=" * 40)
print(f"BEST RMSE: {study_lgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_lgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

# Calculate and print parameter importance
print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_lgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("=" * 40)

# Generate and show visualizations of optimization history,
# parameter importance and parallel coordinate plot
fig1 = plot_optimization_history(study_lgbm)
fig1.show()

fig2 = plot_param_importances(study_lgbm)
fig2.show()

fig3 = plot_parallel_coordinate(
    study_lgbm,
    params=[
        "boosting_type",
        "num_leaves",
        "max_depth",
        "learning_rate",
        "min_split_gain",
        "min_child_weight",
        "min_child_samples",
        "colsample_bytree",
        "reg_alpha",
        "reg_lambda",
        "colsample_bynode",
        "min_data_per_group",
        "max_cat_threshold",
        "cat_l2",
        "cat_smooth",
        "max_cat_to_onehot",
        "max_bin",
        "n_estimators",
    ],
)
fig3.show()

# Save visualizations as HTML files
fig1.write_html(LGBM_TUNING_DIR_REG / "optuna_lgbm_1k_history.html")
fig2.write_html(LGBM_TUNING_DIR_REG / "optuna_lgbm_1k_importance.html")
fig3.write_html(LGBM_TUNING_DIR_REG / "optuna_lgbm_1k_parallel.html")

[I 2026-04-25 21:17:22,745] A new study created in memory with name: no-name-2b44786d-2ca1-42a4-9ca6-313e22e59cab
[I 2026-04-25 21:17:22,906] Trial 1 finished with value: 0.3229374164994242 and parameters: {'boosting_type': 'goss', 'num_leaves': 17, 'max_depth': 3, 'learning_rate': 0.01809795334595728, 'min_split_gain': 6.79410386589513, 'min_child_weight': 0.03540235399406376, 'min_child_samples': 85, 'colsample_bytree': 0.4880070532197824, 'reg_alpha': 0.27600824883479363, 'reg_lambda': 0.3871709661713718, 'colsample_bynode': 0.6002060841825201, 'min_data_per_group': 97, 'max_cat_threshold': 13, 'cat_l2': 0.06740953354613699, 'cat_smooth': 0.10471372018149734, 'max_cat_to_onehot': 1, 'max_bin': 214, 'n_estimators': 193, 'top_rate': 0.19087219030384217, 'other_rate': 0.3342500150313242}. Best is trial 1 with value: 0.3229374164994242.
[I 2026-04-25 21:17:22,933] Trial 7 finished with value: 0.3225886380357504 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 11, 'max_depth': 2, 


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST RMSE: 0.3167
BEST PARAMETERS:
best_params = {
    "boosting_type": "gbdt",
    "num_leaves": 13,
    "max_depth": 3,
    "learning_rate": 0.02169809132057075,
    "min_split_gain": 0.027389027279699385,
    "min_child_weight": 0.6899288118460188,
    "min_child_samples": 42,
    "colsample_bytree": 0.37238284939773186,
    "reg_alpha": 0.09371206330869274,
    "reg_lambda": 0.019391247189888383,
    "colsample_bynode": 0.30902821981225653,
    "min_data_

In [ ]:
# Check overfit on holdout set with best parameters from tuning
best_params = study_lgbm.best_params.copy()

# Add fixed parameters to best_params and print them
best_params["random_state"] = 42
best_params["n_jobs"] = n_cores
best_params["verbose"] = -1
best_params["objective"] = "regression"
best_params["metric"] = "rmse"
print(f"BEST PARAMS: {best_params}")

final_model = LGBMRegressor(**best_params)

# Fit on the entire tuning set
final_model.fit(X_tuning, y_tuning)

# Evaluate on holdout set
holdout_preds = final_model.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

# Print comparison of Optuna CV RMSE and holdout test RMSE
print(f"Optuna Val RMSE: {study_lgbm.best_value:.4f}")
print(f"Holdout Test RMSE: {holdout_rmse:.4f}")

BEST PARAMS: {'boosting_type': 'gbdt', 'num_leaves': 13, 'max_depth': 3, 'learning_rate': 0.02169809132057075, 'min_split_gain': 0.027389027279699385, 'min_child_weight': 0.6899288118460188, 'min_child_samples': 42, 'colsample_bytree': 0.37238284939773186, 'reg_alpha': 0.09371206330869274, 'reg_lambda': 0.019391247189888383, 'colsample_bynode': 0.30902821981225653, 'min_data_per_group': 33, 'max_cat_threshold': 29, 'cat_l2': 0.041250555077113255, 'cat_smooth': 28.95327386652521, 'max_cat_to_onehot': 6, 'max_bin': 161, 'n_estimators': 144, 'subsample': 0.8693945506758759, 'subsample_freq': 5, 'random_state': 42, 'n_jobs': -1, 'verbose': -1, 'objective': 'regression', 'metric': 'rmse'}
Optuna Val RMSE: 0.3167
Holdout Test RMSE: 0.3356


### 10k

In [ ]:
# Parameter tuning settings
timeout_seconds = 60 * 60 * 1
no_improvement_trials = 100

# Split the 10k dataset to evaluate on holdout after tuning
X_tuning = train_10k_X.head(8000)
y_tuning = train_10k_y_reg.head(8000)
X_val_holdout = train_10k_X.tail(2000)
y_val_holdout = train_10k_y_reg.tail(2000)


# Define the objective function for Optuna to optimize
def objective_lgbm(trial):
    # Possible parameters to tune with ranges
    params = {
        "boosting_type": trial.suggest_categorical(
            "boosting_type", ["gbdt", "goss"]
        ),  # Type of boosting algorithm - not using "dart" to save time and "dart" did not deliver significantly better results
        "num_leaves": trial.suggest_int(
            "num_leaves", 3, 31
        ),  # Max leaves per tree. Low to prevent memorizing small data
        "max_depth": trial.suggest_int(
            "max_depth", 2, 8
        ),  # Max tree depth. Low to reduce overfitting
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.2, log=True
        ),  # Step size
        "objective": "regression",  # Regression objective
        "metric": "rmse",  # Evaluation metric
        "min_split_gain": trial.suggest_float(
            "min_split_gain", 0, 10.0
        ),  # Min loss reduction required to make a split
        "min_child_weight": trial.suggest_float(
            "min_child_weight", 1e-3, 5.0, log=True
        ),  # Min sum of instance weight needed in a child node
        "min_child_samples": trial.suggest_int(
            "min_child_samples", 15, 100
        ),  # Min data in one leaf
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.3, 0.8
        ),  # % of features used per tree
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-3, 10.0, log=True
        ),  # L1 regularization
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-3, 10.0, log=True
        ),  # L2 regularization
        "random_state": 42,  # Fixed seed for reproducibility
        "n_jobs": 1,  # Use single thread to avoid issues with parallelism in Optuna
        "verbose": -1,  # Suppress LightGBM output
        "colsample_bynode": trial.suggest_float(
            "colsample_bynode", 0.3, 0.8
        ),  # % of features used per node
        "min_data_per_group": trial.suggest_int(
            "min_data_per_group", 10, 100
        ),  # Min data per categorical group
        "max_cat_threshold": trial.suggest_int(
            "max_cat_threshold", 1, 32
        ),  # Max splits for categories
        "cat_l2": trial.suggest_float(
            "cat_l2", 1e-3, 50.0, log=True
        ),  # L2 regularization for categorical splits
        "cat_smooth": trial.suggest_float(
            "cat_smooth", 1e-3, 50.0, log=True
        ),  # Smoothing for categorical splits
        "max_cat_to_onehot": trial.suggest_int(
            "max_cat_to_onehot", 1, 20
        ),  # Threshold for one-hot encoding
        "max_bin": trial.suggest_int("max_bin", 63, 255),  # Max bins for features
        "n_estimators": trial.suggest_int(
            "n_estimators", 100, 1000
        ),  # Number of boosting rounds
    }

    if params["boosting_type"] == "goss":  # Gradient-based One-Side Sampling (GOSS)
        params["subsample"] = 1.0  # LightGBM requirement for GOSS
        params["top_rate"] = trial.suggest_float(
            "top_rate", 0.05, 0.5
        )  # Fraction of large gradients to retain
        params["other_rate"] = trial.suggest_float(
            "other_rate", 0.05, 1.0 - params["top_rate"]
        )  # Fraction of small gradients to retain
    else:  # GBDT (Gradient Boosting Decision Trees)
        params["subsample"] = trial.suggest_float(
            "subsample", 0.5, 1.0
        )  # Row subsampling
        params["subsample_freq"] = trial.suggest_int(
            "subsample_freq", 1, 5
        )  # Frequency of subsampling

    # TimeSeriesSplit to respect temporal order of data 5 splits for 5-fold CV
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []

    # 5-fold CV cycle
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]

        model = LGBMRegressor(**params)

        # Suppress warnings during fitting
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(X_tr, y_tr)

        # Calculate RMSE
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)

    # Return mean RMSE across CV folds as the objective value to minimize
    return np.mean(cv_scores)


# Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
def stop_optuna(study, trial):
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(
            f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization."
        )
        study.stop()


# Create Optuna study and optimize
study_lgbm = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_lgbm.optimize(
    objective_lgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna]
)

# Print best results and optimal parameters
print("\n" + "=" * 40)
print(f"BEST RMSE: {study_lgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_lgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

# Calculate and print parameter importance
print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_lgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("=" * 40)

# Generate and show visualizations of optimization history,
# parameter importance and parallel coordinate plot
fig1 = plot_optimization_history(study_lgbm)
fig1.show()

fig2 = plot_param_importances(study_lgbm)
fig2.show()

fig3 = plot_parallel_coordinate(
    study_lgbm,
    params=[
        "boosting_type",
        "num_leaves",
        "max_depth",
        "learning_rate",
        "min_split_gain",
        "min_child_weight",
        "min_child_samples",
        "colsample_bytree",
        "reg_alpha",
        "reg_lambda",
        "colsample_bynode",
        "min_data_per_group",
        "max_cat_threshold",
        "cat_l2",
        "cat_smooth",
        "max_cat_to_onehot",
        "max_bin",
        "n_estimators",
    ],
)
fig3.show()

# Save visualizations as HTML files
fig1.write_html(LGBM_TUNING_DIR_REG / "optuna_lgbm_10k_history.html")
fig2.write_html(LGBM_TUNING_DIR_REG / "optuna_lgbm_10k_importance.html")
fig3.write_html(LGBM_TUNING_DIR_REG / "optuna_lgbm_10k_parallel.html")

[I 2026-04-25 21:17:41,737] A new study created in memory with name: no-name-5386ba86-6bee-4c6b-8db3-37c99fe6b755
[I 2026-04-25 21:17:42,402] Trial 7 finished with value: 0.3005803681127669 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 12, 'max_depth': 2, 'learning_rate': 0.0829900058557435, 'min_split_gain': 9.979720271571798, 'min_child_weight': 0.44732746424149855, 'min_child_samples': 76, 'colsample_bytree': 0.3210188344379993, 'reg_alpha': 0.18244765042072356, 'reg_lambda': 0.007628627024403479, 'colsample_bynode': 0.5319244690302944, 'min_data_per_group': 35, 'max_cat_threshold': 15, 'cat_l2': 0.00156953670110347, 'cat_smooth': 0.0021630217067049916, 'max_cat_to_onehot': 8, 'max_bin': 149, 'n_estimators': 118, 'subsample': 0.8453660172644109, 'subsample_freq': 5}. Best is trial 7 with value: 0.3005803681127669.
[I 2026-04-25 21:17:42,430] Trial 2 finished with value: 0.3004901827126455 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 3, 'max_depth': 5, 'learning_


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 21:18:36,513] Trial 152 finished with value: 0.29807428577901557 and parameters: {'boosting_type': 'goss', 'num_leaves': 29, 'max_depth': 5, 'learning_rate': 0.01019880880179556, 'min_split_gain': 0.4517020287798905, 'min_child_weight': 0.0316308633253871, 'min_child_samples': 91, 'colsample_bytree': 0.5625865658628485, 'reg_alpha': 0.6239122532749615, 'reg_lambda': 8.087784734418108, 'colsample_bynode': 0.7218487560540257, 'min_data_per_group': 96, 'max_cat_threshold': 3, 'cat_l2': 0.21572365698821777, 'cat_smooth': 0.02160618360469681, 'max_cat_to_onehot': 20, 'max_bin': 238, 'n_estimators': 289, 'top_rate': 0.1103281159035176, 'other_rate': 0.86852575725831}. Best is trial 48 with value: 0.2972893499643337.
[I 2026-04-25 21:18:36,648] Trial 149 finished with value: 0.2980816767822706 and parameters: {'boosting_type': 'goss', 'num_leaves': 26, 'max_depth': 5, 'learning_rate': 0.013618112509908197, 'min_split_gain': 0.23960638567836684, 'min_child_weight': 0.003055030253


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 21:18:36,867] Trial 151 finished with value: 0.2987996565803447 and parameters: {'boosting_type': 'goss', 'num_leaves': 26, 'max_depth': 5, 'learning_rate': 0.007906419475402706, 'min_split_gain': 0.37818102496003225, 'min_child_weight': 0.08035749271577715, 'min_child_samples': 92, 'colsample_bytree': 0.41334138294270145, 'reg_alpha': 8.226492207836996, 'reg_lambda': 7.478905954999174, 'colsample_bynode': 0.7233099875756572, 'min_data_per_group': 86, 'max_cat_threshold': 3, 'cat_l2': 8.640456874174935, 'cat_smooth': 0.021611885096309888, 'max_cat_to_onehot': 20, 'max_bin': 252, 'n_estimators': 587, 'top_rate': 0.11803587315919983, 'other_rate': 0.8680489199756264}. Best is trial 48 with value: 0.2972893499643337.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 21:18:37,082] Trial 150 finished with value: 0.29789052353938283 and parameters: {'boosting_type': 'goss', 'num_leaves': 26, 'max_depth': 5, 'learning_rate': 0.008039816073854117, 'min_split_gain': 0.39535342856897937, 'min_child_weight': 0.06740816819561234, 'min_child_samples': 94, 'colsample_bytree': 0.5652638172910918, 'reg_alpha': 0.49447892695439394, 'reg_lambda': 0.0016790001669628456, 'colsample_bynode': 0.7249531374565445, 'min_data_per_group': 81, 'max_cat_threshold': 3, 'cat_l2': 0.23726182715785787, 'cat_smooth': 0.01208284325692807, 'max_cat_to_onehot': 20, 'max_bin': 244, 'n_estimators': 584, 'top_rate': 0.11415204416726853, 'other_rate': 0.8696122178589257}. Best is trial 48 with value: 0.2972893499643337.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 21:18:37,321] Trial 153 finished with value: 0.29838085775255346 and parameters: {'boosting_type': 'goss', 'num_leaves': 29, 'max_depth': 5, 'learning_rate': 0.01093157606718395, 'min_split_gain': 0.48098655288072023, 'min_child_weight': 0.28762100814366853, 'min_child_samples': 92, 'colsample_bytree': 0.5683832762308623, 'reg_alpha': 0.5254540772479291, 'reg_lambda': 0.0028040752883619234, 'colsample_bynode': 0.6931962273173679, 'min_data_per_group': 96, 'max_cat_threshold': 32, 'cat_l2': 0.19792993039747042, 'cat_smooth': 0.022675040924014452, 'max_cat_to_onehot': 20, 'max_bin': 244, 'n_estimators': 595, 'top_rate': 0.11265803622960016, 'other_rate': 0.7353763595178933}. Best is trial 48 with value: 0.2972893499643337.



[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST RMSE: 0.2973
BEST PARAMETERS:
best_params = {
    "boosting_type": "goss",
    "num_leaves": 28,
    "max_depth": 3,
    "learning_rate": 0.01147519853547454,
    "min_split_gain": 0.041494825179870576,
    "min_child_weight": 0.012349230729538438,
    "min_child_samples": 81,
    "colsample_bytree": 0.5283693462952471,
    "reg_alpha": 9.158641225780748,
    "reg_lambda": 0.004093641172128938,
    "colsample_bynode": 0.6725807827106883,
    "min_data_per_group": 91,
    "max_cat_threshold": 6,
    "cat_l2": 1.5908743669801766,
    "cat_smooth": 0.14729665520231705,
    "max_cat_to_onehot": 11,
    "max_bin": 156,
    "n_estimators": 747,
    "top_rate": 0.1092716955430614,
    "other_rate": 0.777449860751789,
}

--- PARAMETER IMPORTANCE ---
  min_split_gain      : 0.8956
  max_cat_threshold   : 0.0359
  colsample_bynode    : 0.0239
  num_leaves          : 0.0139
  cat_smooth          : 0.0083
  n_estimators      

In [ ]:
# Check overfit on holdout set with best parameters from tuning
best_params = study_lgbm.best_params.copy()

# Apply scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]
best_params["n_estimators"] = original_trees * 2
best_params["learning_rate"] = original_lr / 2
print(
    f"\n[Scaling Trick Applied] {best_params['boosting_type'].upper()}: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}"
)

# Add fixed parameters to best_params and print them
best_params["random_state"] = 42
best_params["n_jobs"] = n_cores
best_params["verbose"] = -1
best_params["objective"] = "regression"
best_params["metric"] = "rmse"
print(f"BEST PARAMS: {best_params}")

final_model = LGBMRegressor(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning)

# Evaluate on holdout set
holdout_preds = final_model.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

# Print comparison of Optuna CV RMSE and holdout test RMSE
print("\n" + "=" * 40)
print(f"Optuna Val RMSE:   {study_lgbm.best_value:.4f}")
print(f"Holdout Test RMSE: {holdout_rmse:.4f}")
print("=" * 40)


[Scaling Trick Applied] GOSS: Trees 747 -> 1494, LR 0.0115 -> 0.0057
BEST PARAMS: {'boosting_type': 'goss', 'num_leaves': 28, 'max_depth': 3, 'learning_rate': 0.00573759926773727, 'min_split_gain': 0.041494825179870576, 'min_child_weight': 0.012349230729538438, 'min_child_samples': 81, 'colsample_bytree': 0.5283693462952471, 'reg_alpha': 9.158641225780748, 'reg_lambda': 0.004093641172128938, 'colsample_bynode': 0.6725807827106883, 'min_data_per_group': 91, 'max_cat_threshold': 6, 'cat_l2': 1.5908743669801766, 'cat_smooth': 0.14729665520231705, 'max_cat_to_onehot': 11, 'max_bin': 156, 'n_estimators': 1494, 'top_rate': 0.1092716955430614, 'other_rate': 0.777449860751789, 'random_state': 42, 'n_jobs': -1, 'verbose': -1, 'objective': 'regression', 'metric': 'rmse'}

Optuna Val RMSE:   0.2973
Holdout Test RMSE: 0.3126


### 100k

In [ ]:
# Parameter tuning settings
timeout_seconds = 60 * 60 * 3
no_improvement_trials = 100

# Split the 100k dataset to evaluate on holdout after tuning
X_tuning = train_100k_X.head(80000)
y_tuning = train_100k_y_reg.head(80000)
X_val_holdout = train_100k_X.tail(20000)
y_val_holdout = train_100k_y_reg.tail(20000)


# Define the objective function for Optuna to optimize
def objective_lgbm(trial):
    params = {
        "boosting_type": trial.suggest_categorical(
            "boosting_type", ["gbdt", "goss"]
        ),  # Type of boosting algorithm - not using "dart" to save time and "dart" did not deliver significantly better results
        "num_leaves": trial.suggest_int("num_leaves", 5, 512),  # Max leaves per tree
        "max_depth": trial.suggest_int("max_depth", -1, 20),  # Max tree depth
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.2, log=True
        ),  # Step size
        "objective": "regression",  # Regression objective
        "metric": "rmse",  # Evaluation metric
        "min_split_gain": trial.suggest_float(
            "min_split_gain", 0, 30.0
        ),  # Min loss reduction required to make a split
        "min_child_weight": trial.suggest_float(
            "min_child_weight", 1e-5, 10.0, log=True
        ),  # Min sum of instance weight needed in a child node
        "min_child_samples": trial.suggest_int(
            "min_child_samples", 10, 500
        ),  # Min data in one leaf
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.2, 1.0
        ),  # % of features used per tree
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-8, 100.0, log=True
        ),  # L1 regularization
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-8, 100.0, log=True
        ),  # L2 regularization
        "random_state": 42,  # Fixed seed for reproducibility
        "n_jobs": 1,  # Use single thread to avoid issues with parallelism in Optuna
        "verbose": -1,  # Suppress LightGBM output
        "colsample_bynode": trial.suggest_float(
            "colsample_bynode", 0.2, 1.0
        ),  # % of features used per node
        "min_data_per_group": trial.suggest_int(
            "min_data_per_group", 1, 1000
        ),  # Min data per categorical group
        "max_cat_threshold": trial.suggest_int(
            "max_cat_threshold", 1, 1000
        ),  # Max splits for categories
        "cat_l2": trial.suggest_float(
            "cat_l2", 1e-8, 100.0, log=True
        ),  # L2 regularization for categorical splits
        "cat_smooth": trial.suggest_float(
            "cat_smooth", 1e-8, 100.0, log=True
        ),  # Smoothing for categorical splits
        "max_cat_to_onehot": trial.suggest_int(
            "max_cat_to_onehot", 1, 51
        ),  # Threshold for one-hot encoding
        "max_bin": trial.suggest_int("max_bin", 63, 511),  # Max bins for features
        "n_estimators": trial.suggest_int(
            "n_estimators", 100, 1000
        ),  # Number of boosting rounds
    }

    if params["boosting_type"] == "goss":  # Gradient-based One-Side Sampling (GOSS)
        params["subsample"] = 1.0  # LightGBM requirement for GOSS
        params["top_rate"] = trial.suggest_float(
            "top_rate", 0.05, 0.8
        )  # Fraction of large gradients to retain
        params["other_rate"] = trial.suggest_float(
            "other_rate", 0.05, 1.0 - params["top_rate"]
        )  # Fraction of small gradients to retain
    else:
        params["subsample"] = trial.suggest_float(
            "subsample", 0.1, 1.0
        )  # Row subsampling
        params["subsample_freq"] = trial.suggest_int(
            "subsample_freq", 1, 10
        )  # Frequency of subsampling

    # TimeSeriesSplit to respect temporal order of data 5 splits for 5-fold CV
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []

    # 5-fold CV cycle
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]

        model = LGBMRegressor(**params)

        # Suppress warnings during fitting
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(X_tr, y_tr)

        # Calculate RMSE
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)

    # Return mean RMSE across CV folds as the objective value to minimize
    return np.mean(cv_scores)


# Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
def stop_optuna(study, trial):
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(
            f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization."
        )
        study.stop()


# Create Optuna study and optimize
study_lgbm = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_lgbm.optimize(
    objective_lgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna]
)

# Print best results and optimal parameters
print("\n" + "=" * 40)
print(f"BEST RMSE: {study_lgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_lgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

# Calculate and print parameter importance
print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_lgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("=" * 40)

# Generate and show visualizations of optimization history,
# parameter importance and parallel coordinate plot
fig1 = plot_optimization_history(study_lgbm)
fig1.show()

fig2 = plot_param_importances(study_lgbm)
fig2.show()

fig3 = plot_parallel_coordinate(
    study_lgbm,
    params=[
        "boosting_type",
        "num_leaves",
        "max_depth",
        "learning_rate",
        "min_split_gain",
        "min_child_weight",
        "min_child_samples",
        "colsample_bytree",
        "reg_alpha",
        "reg_lambda",
        "colsample_bynode",
        "min_data_per_group",
        "max_cat_threshold",
        "cat_l2",
        "cat_smooth",
        "max_cat_to_onehot",
        "max_bin",
        "n_estimators",
    ],
)
fig3.show()

# Save visualizations as HTML files
fig1.write_html(LGBM_TUNING_DIR_REG / "optuna_lgbm_100k_history.html")
fig2.write_html(LGBM_TUNING_DIR_REG / "optuna_lgbm_100k_importance.html")
fig3.write_html(LGBM_TUNING_DIR_REG / "optuna_lgbm_100k_parallel.html")

[I 2026-04-25 21:18:41,415] A new study created in memory with name: no-name-35dffbaa-11e4-4bfc-99ea-e6555009813b
[I 2026-04-25 21:18:49,479] Trial 6 finished with value: 0.3082582147156724 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 433, 'max_depth': 6, 'learning_rate': 0.005033252023003456, 'min_split_gain': 27.69876502744986, 'min_child_weight': 0.030529693502711203, 'min_child_samples': 86, 'colsample_bytree': 0.9295094820970584, 'reg_alpha': 0.16575046979455088, 'reg_lambda': 0.0005648312750290265, 'colsample_bynode': 0.23251139034753754, 'min_data_per_group': 766, 'max_cat_threshold': 492, 'cat_l2': 0.0003407262829256871, 'cat_smooth': 0.09909254138246734, 'max_cat_to_onehot': 46, 'max_bin': 278, 'n_estimators': 456, 'subsample': 0.306005985922797, 'subsample_freq': 6}. Best is trial 6 with value: 0.3082582147156724.
[I 2026-04-25 21:18:51,370] Trial 2 finished with value: 0.3047038473198146 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 85, 'max_depth': 17, 


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 21:28:29,310] Trial 139 finished with value: 0.30033891049553574 and parameters: {'boosting_type': 'goss', 'num_leaves': 131, 'max_depth': 3, 'learning_rate': 0.016874520923609877, 'min_split_gain': 0.4631420764054974, 'min_child_weight': 0.040673643352331194, 'min_child_samples': 212, 'colsample_bytree': 0.6961401596923971, 'reg_alpha': 0.00012135875976024526, 'reg_lambda': 1.3750191915121786e-07, 'colsample_bynode': 0.38700309092606194, 'min_data_per_group': 823, 'max_cat_threshold': 116, 'cat_l2': 0.0018789498424281864, 'cat_smooth': 2.109829812451524e-08, 'max_cat_to_onehot': 25, 'max_bin': 411, 'n_estimators': 508, 'top_rate': 0.5774494682164257, 'other_rate': 0.14023057345238144}. Best is trial 40 with value: 0.29941925086857446.
[I 2026-04-25 21:28:37,582] Trial 141 finished with value: 0.30052458946702654 and parameters: {'boosting_type': 'goss', 'num_leaves': 105, 'max_depth': 3, 'learning_rate': 0.015046519328731077, 'min_split_gain': 0.46521838140264465, 'min_c


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 21:28:39,347] Trial 138 finished with value: 0.2998683794989687 and parameters: {'boosting_type': 'goss', 'num_leaves': 495, 'max_depth': 16, 'learning_rate': 0.014749428203931169, 'min_split_gain': 0.5123089470810481, 'min_child_weight': 6.581300984867822e-05, 'min_child_samples': 127, 'colsample_bytree': 0.751041251737995, 'reg_alpha': 0.0002914333983895987, 'reg_lambda': 0.0014925056073495436, 'colsample_bynode': 0.3790867241009994, 'min_data_per_group': 832, 'max_cat_threshold': 194, 'cat_l2': 0.002136080903479217, 'cat_smooth': 1.100476773061757e-07, 'max_cat_to_onehot': 25, 'max_bin': 334, 'n_estimators': 513, 'top_rate': 0.7616920851820701, 'other_rate': 0.08686758498611864}. Best is trial 40 with value: 0.29941925086857446.
[I 2026-04-25 21:28:45,780] Trial 142 finished with value: 0.30013600074121816 and parameters: {'boosting_type': 'goss', 'num_leaves': 150, 'max_depth': 4, 'learning_rate': 0.014736683545050764, 'min_split_gain': 0.5554730360147684, 'min_child_


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 21:28:46,895] Trial 143 finished with value: 0.30042851259475684 and parameters: {'boosting_type': 'goss', 'num_leaves': 68, 'max_depth': 3, 'learning_rate': 0.014707694216410709, 'min_split_gain': 0.579745684849255, 'min_child_weight': 0.05707934189680624, 'min_child_samples': 159, 'colsample_bytree': 0.6608213848245409, 'reg_alpha': 0.0001378890570924243, 'reg_lambda': 2.0482712609907417e-06, 'colsample_bynode': 0.31580394999199096, 'min_data_per_group': 892, 'max_cat_threshold': 224, 'cat_l2': 0.03158766312734346, 'cat_smooth': 2.1203365670427546e-08, 'max_cat_to_onehot': 24, 'max_bin': 392, 'n_estimators': 556, 'top_rate': 0.5740140835461194, 'other_rate': 0.1391347088782899}. Best is trial 40 with value: 0.29941925086857446.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-25 21:28:55,762] Trial 132 finished with value: 0.3009266147301132 and parameters: {'boosting_type': 'goss', 'num_leaves': 211, 'max_depth': 15, 'learning_rate': 0.021093026570354296, 'min_split_gain': 0.04344958278347058, 'min_child_weight': 0.014884499666239339, 'min_child_samples': 69, 'colsample_bytree': 0.6978346720413228, 'reg_alpha': 0.00030132820648735365, 'reg_lambda': 0.001072274259866118, 'colsample_bynode': 0.2919615788440586, 'min_data_per_group': 521, 'max_cat_threshold': 201, 'cat_l2': 0.0007818959557443085, 'cat_smooth': 23.331811210634434, 'max_cat_to_onehot': 27, 'max_bin': 469, 'n_estimators': 698, 'top_rate': 0.7666566178007008, 'other_rate': 0.08868984942180758}. Best is trial 40 with value: 0.29941925086857446.
[I 2026-04-25 21:29:02,907] Trial 133 finished with value: 0.3001416130433367 and parameters: {'boosting_type': 'goss', 'num_leaves': 435, 'max_depth': 15, 'learning_rate': 0.017355500882304455, 'min_split_gain': 0.02632997827816813, 'min_child_w


BEST RMSE: 0.2994
BEST PARAMETERS:
best_params = {
    "boosting_type": "gbdt",
    "num_leaves": 321,
    "max_depth": 7,
    "learning_rate": 0.01495737541649921,
    "min_split_gain": 0.15169683434030254,
    "min_child_weight": 0.0043443738122844215,
    "min_child_samples": 129,
    "colsample_bytree": 0.5703676576888828,
    "reg_alpha": 0.0009598013297060651,
    "reg_lambda": 1.512993041670286e-06,
    "colsample_bynode": 0.40283945018201495,
    "min_data_per_group": 572,
    "max_cat_threshold": 557,
    "cat_l2": 0.1829638573103202,
    "cat_smooth": 72.95247318237497,
    "max_cat_to_onehot": 51,
    "max_bin": 341,
    "n_estimators": 674,
    "subsample": 0.998415655290472,
    "subsample_freq": 7,
}

--- PARAMETER IMPORTANCE ---
  min_split_gain      : 0.8810
  min_child_samples   : 0.0385
  colsample_bytree    : 0.0178
  max_bin             : 0.0143
  learning_rate       : 0.0130
  reg_alpha           : 0.0088
  max_cat_threshold   : 0.0082
  max_cat_to_onehot   : 0.00

In [ ]:
# Check overfit on holdout set with best parameters from tuning
best_params = study_lgbm.best_params.copy()

# Apply scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]
best_params["n_estimators"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10
print(
    f"\n[Scaling Trick Applied] {best_params['boosting_type'].upper()}: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}"
)

# Add fixed parameters to best_params and print them
best_params["random_state"] = 42
best_params["n_jobs"] = n_cores
best_params["verbose"] = -1
best_params["objective"] = "regression"
best_params["metric"] = "rmse"
print(f"BEST PARAMS: {best_params}")

final_model = LGBMRegressor(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning)

# Evaluate on holdout set
holdout_preds = final_model.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

# Print comparison of Optuna CV RMSE and holdout test RMSE
print("\n" + "=" * 40)
print(f"Optuna Val RMSE:   {study_lgbm.best_value:.4f}")
print(f"Holdout Test RMSE: {holdout_rmse:.4f}")
print("=" * 40)


[Scaling Trick Applied] GBDT: Trees 674 -> 6740, LR 0.0150 -> 0.0015
BEST PARAMS: {'boosting_type': 'gbdt', 'num_leaves': 321, 'max_depth': 7, 'learning_rate': 0.0014957375416499211, 'min_split_gain': 0.15169683434030254, 'min_child_weight': 0.0043443738122844215, 'min_child_samples': 129, 'colsample_bytree': 0.5703676576888828, 'reg_alpha': 0.0009598013297060651, 'reg_lambda': 1.512993041670286e-06, 'colsample_bynode': 0.40283945018201495, 'min_data_per_group': 572, 'max_cat_threshold': 557, 'cat_l2': 0.1829638573103202, 'cat_smooth': 72.95247318237497, 'max_cat_to_onehot': 51, 'max_bin': 341, 'n_estimators': 6740, 'subsample': 0.998415655290472, 'subsample_freq': 7, 'random_state': 42, 'n_jobs': -1, 'verbose': -1, 'objective': 'regression', 'metric': 'rmse'}

Optuna Val RMSE:   0.2994
Holdout Test RMSE: 0.2985


### Whole training data set

In [ ]:
# Parameter tuning settings
timeout_seconds = 60 * 60 * 4
no_improvement_trials = 100

# Split the dataset to evaluate on holdout after tuning
split_index = int(len(train_full_X) * 0.8)
X_tuning = train_full_X[:split_index]
y_tuning = train_full_y_reg[:split_index]
X_val_holdout = train_full_X[split_index:]
y_val_holdout = train_full_y_reg[split_index:]


# Define the objective function for Optuna to optimize
def objective_lgbm(trial):
    # Possible parameters to tune with ranges
    params = {
        "boosting_type": trial.suggest_categorical(
            "boosting_type", ["gbdt", "goss"]
        ),  # Type of boosting algorithm - not using "dart" to save time and "dart" did not deliver significantly better results
        "num_leaves": trial.suggest_int("num_leaves", 5, 512),  # Max leaves per tree
        "max_depth": trial.suggest_int("max_depth", -1, 20),  # Max tree depth
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.2, log=True
        ),  # Step size
        "objective": "regression",  # Regression objective
        "metric": "rmse",  # Evaluation metric
        "min_split_gain": trial.suggest_float(
            "min_split_gain", 0, 30.0
        ),  # Min loss reduction required to make a split
        "min_child_weight": trial.suggest_float(
            "min_child_weight", 1e-5, 10.0, log=True
        ),  # Min sum of instance weight needed in a child node
        "min_child_samples": trial.suggest_int(
            "min_child_samples", 10, 500
        ),  # Min data in one leaf
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.2, 1.0
        ),  # % of features used per tree
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-8, 100.0, log=True
        ),  # L1 regularization
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-8, 100.0, log=True
        ),  # L2 regularization
        "random_state": 42,  # Fixed seed for reproducibility
        "n_jobs": 1,  # Use single thread to avoid issues with parallelism in Optuna
        "verbose": -1,  # Suppress LightGBM output
        "colsample_bynode": trial.suggest_float(
            "colsample_bynode", 0.2, 1.0
        ),  # % of features used per node
        "min_data_per_group": trial.suggest_int(
            "min_data_per_group", 1, 1000
        ),  # Min data per categorical group
        "max_cat_threshold": trial.suggest_int(
            "max_cat_threshold", 1, 1000
        ),  # Max splits for categories
        "cat_l2": trial.suggest_float(
            "cat_l2", 1e-8, 100.0, log=True
        ),  # L2 regularization for categorical splits
        "cat_smooth": trial.suggest_float(
            "cat_smooth", 1e-8, 100.0, log=True
        ),  # Smoothing for categorical splits
        "max_cat_to_onehot": trial.suggest_int(
            "max_cat_to_onehot", 1, 51
        ),  # Threshold for one-hot encoding
        "max_bin": trial.suggest_int("max_bin", 63, 511),  # Max bins for features
        "n_estimators": trial.suggest_int(
            "n_estimators", 100, 1000
        ),  # Number of boosting rounds
    }

    if params["boosting_type"] == "goss":  # Gradient-based One-Side Sampling (GOSS)
        params["subsample"] = 1.0  # LightGBM requirement for GOSS
        params["top_rate"] = trial.suggest_float(
            "top_rate", 0.05, 0.8
        )  # Fraction of large gradients to retain
        params["other_rate"] = trial.suggest_float(
            "other_rate", 0.05, 1.0 - params["top_rate"]
        )  # Fraction of small gradients to retain
    else:
        params["subsample"] = trial.suggest_float(
            "subsample", 0.1, 1.0
        )  # Row subsampling
        params["subsample_freq"] = trial.suggest_int(
            "subsample_freq", 1, 10
        )  # Frequency of subsampling

    # TimeSeriesSplit to respect temporal order of data 3 splits for 3-fold CV
    tscv = TimeSeriesSplit(n_splits=3)
    cv_scores = []

    # 3-fold CV cycle
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]

        model = LGBMRegressor(**params)

        # Suppress warnings during fitting
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(X_tr, y_tr)

        # Calculate RMSE
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)

    # Return mean RMSE across CV folds as the objective value to minimize
    return np.mean(cv_scores)


# Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
def stop_optuna(study, trial):
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(
            f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization."
        )
        study.stop()


# Create Optuna study and optimize
study_lgbm = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_lgbm.optimize(
    objective_lgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna]
)

# Print best results and optimal parameters
print("\n" + "=" * 40)
print(f"BEST RMSE: {study_lgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_lgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

# Calculate and print parameter importance
print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_lgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("=" * 40)

# Generate and show visualizations of optimization history,
# parameter importance and parallel coordinate plot
fig1 = plot_optimization_history(study_lgbm)
fig1.show()

fig2 = plot_param_importances(study_lgbm)
fig2.show()

fig3 = plot_parallel_coordinate(
    study_lgbm,
    params=[
        "boosting_type",
        "num_leaves",
        "max_depth",
        "learning_rate",
        "min_split_gain",
        "min_child_weight",
        "min_child_samples",
        "colsample_bytree",
        "reg_alpha",
        "reg_lambda",
        "colsample_bynode",
        "min_data_per_group",
        "max_cat_threshold",
        "cat_l2",
        "cat_smooth",
        "max_cat_to_onehot",
        "max_bin",
        "n_estimators",
    ],
)
fig3.show()

# Save visualizations as HTML files
fig1.write_html(LGBM_TUNING_DIR_REG / "optuna_lgbm_full_history.html")
fig2.write_html(LGBM_TUNING_DIR_REG / "optuna_lgbm_full_importance.html")
fig3.write_html(LGBM_TUNING_DIR_REG / "optuna_lgbm_full_parallel.html")

[I 2026-04-26 14:32:01,070] A new study created in memory with name: no-name-a2aa9d3c-0a4f-44c1-9e7c-9fa45773a751
[I 2026-04-26 14:32:44,901] Trial 2 finished with value: 0.23654120249672103 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 316, 'max_depth': 8, 'learning_rate': 0.13730947267204555, 'min_split_gain': 4.828837048447326, 'min_child_weight': 0.0006294934164022466, 'min_child_samples': 215, 'colsample_bytree': 0.8853581334862928, 'reg_alpha': 0.0008748970529039687, 'reg_lambda': 1.303888030528829e-05, 'colsample_bynode': 0.526960671752362, 'min_data_per_group': 560, 'max_cat_threshold': 855, 'cat_l2': 0.0025192791437769293, 'cat_smooth': 0.03560049274072963, 'max_cat_to_onehot': 50, 'max_bin': 114, 'n_estimators': 946, 'subsample': 0.19555988600108073, 'subsample_freq': 7}. Best is trial 2 with value: 0.23654120249672103.
[I 2026-04-26 14:33:07,827] Trial 3 finished with value: 0.23598331156914312 and parameters: {'boosting_type': 'goss', 'num_leaves': 200, 'max_depth


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 17:07:32,559] Trial 310 finished with value: 0.2335086887408597 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 326, 'max_depth': 8, 'learning_rate': 0.036307022121065305, 'min_split_gain': 0.5881555181998578, 'min_child_weight': 0.007375775270957496, 'min_child_samples': 457, 'colsample_bytree': 0.9183833617253395, 'reg_alpha': 0.8043203071086178, 'reg_lambda': 1.5160975933640532e-08, 'colsample_bynode': 0.5863460963565188, 'min_data_per_group': 747, 'max_cat_threshold': 521, 'cat_l2': 0.0003058133661662301, 'cat_smooth': 64.07342359708834, 'max_cat_to_onehot': 48, 'max_bin': 83, 'n_estimators': 597, 'subsample': 0.9746676210563945, 'subsample_freq': 4}. Best is trial 211 with value: 0.2326647143148516.
[I 2026-04-26 17:07:34,241] Trial 308 finished with value: 0.2334744796145963 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 330, 'max_depth': 10, 'learning_rate': 0.03666439770343056, 'min_split_gain': 0.5724444717951019, 'min_child_weight': 0.0358812295


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 17:09:08,471] Trial 314 finished with value: 0.23344861287774743 and parameters: {'boosting_type': 'goss', 'num_leaves': 337, 'max_depth': 6, 'learning_rate': 0.03352383965105839, 'min_split_gain': 0.5921934918620652, 'min_child_weight': 0.005981303340954152, 'min_child_samples': 500, 'colsample_bytree': 0.9266787677237917, 'reg_alpha': 0.05005045668647664, 'reg_lambda': 1.953596166527471e-08, 'colsample_bynode': 0.5571748604383971, 'min_data_per_group': 430, 'max_cat_threshold': 519, 'cat_l2': 0.00012829806269821898, 'cat_smooth': 18.514730232266807, 'max_cat_to_onehot': 46, 'max_bin': 95, 'n_estimators': 609, 'top_rate': 0.11861363931566793, 'other_rate': 0.8659488484158588}. Best is trial 211 with value: 0.2326647143148516.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 17:10:00,436] Trial 303 finished with value: 0.23282166941856475 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 327, 'max_depth': 8, 'learning_rate': 0.03670805352067951, 'min_split_gain': 0.003930455720294858, 'min_child_weight': 0.0033185967507950464, 'min_child_samples': 470, 'colsample_bytree': 0.9470084446869789, 'reg_alpha': 0.08002263577304966, 'reg_lambda': 1.6370119252754567e-08, 'colsample_bynode': 0.5554346473524603, 'min_data_per_group': 618, 'max_cat_threshold': 325, 'cat_l2': 0.002250991680869346, 'cat_smooth': 6.171123805552418, 'max_cat_to_onehot': 38, 'max_bin': 63, 'n_estimators': 605, 'subsample': 0.9551110944104331, 'subsample_freq': 2}. Best is trial 211 with value: 0.2326647143148516.
[I 2026-04-26 17:11:16,681] Trial 315 finished with value: 0.23275690778107525 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 339, 'max_depth': 6, 'learning_rate': 0.03367486455578574, 'min_split_gain': 0.006695665347444202, 'min_child_weight': 1.34922


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 17:11:33,800] Trial 313 finished with value: 0.23277973111589825 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 331, 'max_depth': 8, 'learning_rate': 0.03395533833121295, 'min_split_gain': 0.02175880869798895, 'min_child_weight': 0.005400445298788943, 'min_child_samples': 500, 'colsample_bytree': 0.922841435925934, 'reg_alpha': 0.8530466207324036, 'reg_lambda': 1.639857641570791e-08, 'colsample_bynode': 0.5593078108471672, 'min_data_per_group': 640, 'max_cat_threshold': 520, 'cat_l2': 0.00015848196814328034, 'cat_smooth': 17.74237226381459, 'max_cat_to_onehot': 46, 'max_bin': 95, 'n_estimators': 601, 'subsample': 0.9770453792425484, 'subsample_freq': 4}. Best is trial 211 with value: 0.2326647143148516.



[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST RMSE: 0.2327
BEST PARAMETERS:
best_params = {
    "boosting_type": "gbdt",
    "num_leaves": 367,
    "max_depth": 9,
    "learning_rate": 0.037106925990830715,
    "min_split_gain": 0.028541720097881224,
    "min_child_weight": 3.114973690234082e-05,
    "min_child_samples": 449,
    "colsample_bytree": 0.9583470008702835,
    "reg_alpha": 4.042104970937467,
    "reg_lambda": 4.013246511788023e-08,
    "colsample_bynode": 0.544229805808371,
    "min_data_per_group": 745,
    "max_cat_threshold": 240,
    "cat_l2": 1.4174693320549176e-05,
    "cat_smooth": 0.006490614096342806,
    "max_cat_to_onehot": 4,
    "max_bin": 97,
    "n_estimators": 594,
    "subsample": 0.9848817124796165,
    "subsample_freq": 7,
}

--- PARAMETER IMPORTANCE ---
  min_split_gain      : 0.9129
  min_data_per_group  : 0.0191
  learning_rate       : 0.0190
  max_cat_threshold   : 0.0099
  max_bin             : 0.0091
  colsample_bytree   

In [ ]:
# Check overfit on holdout set with best parameters from tuning
best_params = study_lgbm.best_params.copy()

# Apply scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]
best_params["n_estimators"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10
print(f"\n[Scaling Trick Applied] {best_params['boosting_type'].upper()}: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters to best_params and print them
best_params["random_state"] = 42
best_params["n_jobs"] = n_cores
best_params["verbose"] = -1
best_params["objective"] = "regression"
best_params["metric"] = "rmse"
print(f"BEST PARAMS: {best_params}")

final_model = LGBMRegressor(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning)

# Evaluate on holdout set
holdout_preds = final_model.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

# Print comparison of Optuna CV RMSE and holdout test RMSE
print("\n" + "="*40)
print(f"Optuna Val RMSE:   {study_lgbm.best_value:.4f}")
print(f"Holdout Test RMSE: {holdout_rmse:.4f}")
print("="*40)


[Scaling Trick Applied] GBDT: Trees 594 -> 5940, LR 0.0371 -> 0.0037
BEST PARAMS: {'boosting_type': 'gbdt', 'num_leaves': 367, 'max_depth': 9, 'learning_rate': 0.0037106925990830716, 'min_split_gain': 0.028541720097881224, 'min_child_weight': 3.114973690234082e-05, 'min_child_samples': 449, 'colsample_bytree': 0.9583470008702835, 'reg_alpha': 4.042104970937467, 'reg_lambda': 4.013246511788023e-08, 'colsample_bynode': 0.544229805808371, 'min_data_per_group': 745, 'max_cat_threshold': 240, 'cat_l2': 1.4174693320549176e-05, 'cat_smooth': 0.006490614096342806, 'max_cat_to_onehot': 4, 'max_bin': 97, 'n_estimators': 5940, 'subsample': 0.9848817124796165, 'subsample_freq': 7, 'random_state': 42, 'n_jobs': -1, 'verbose': -1, 'objective': 'regression', 'metric': 'rmse'}

Optuna Val RMSE:   0.2327
Holdout Test RMSE: 0.2835
